In [ ]:
# =============================================================================
# WIMBLEDON 2026 ANALYTICS — MASTER NOTEBOOK
# What Actually Wins Wimbledon? | Fanis Vyronos | LSE
# =============================================================================
# Run this notebook top to bottom. Each section is self-contained and builds
# on the previous one. Sections are separated by large comment headers so you
# can split them into separate Jupyter cells if you prefer.
# =============================================================================

In [ ]:
# SECTION 1 — IMPORTS
# ===============================================================================
import pandas as pd
import numpy as np
import random
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as scipy_stats
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_curve, auc
from xgboost import XGBClassifier

print("✅ Imports loaded")

In [ ]:
# SECTION 2 — HISTORICAL DATA: 2019
# ===============================================================================
players_2019 = [
    "Mikhail Kukushkin", "Guido Pella", "Roberto Bautista Agut", "Milos Raonic",
    "Roger Federer", "Novak Djokovic", "Kei Nishikori", "Tennys Sandgren",
    "Fernando Verdasco", "Rafael Nadal", "Sam Querrey", "Joao Sousa",
    "Ugo Humbert", "Matteo Berrettini", "Benoit Paire", "David Goffin"
]
df_2019 = pd.DataFrame({'player': players_2019})

df_2019['first_serve_pct'] = df_2019['player'].map({
    "Mikhail Kukushkin": 72.71, "Guido Pella": 69.73, "Roberto Bautista Agut": 68.05,
    "Milos Raonic": 66.67, "Roger Federer": 64.82, "Novak Djokovic": 64.27,
    "Kei Nishikori": 64.19, "Tennys Sandgren": 63.64, "Fernando Verdasco": 61.86,
    "Rafael Nadal": 61.85, "Sam Querrey": 61.66, "Joao Sousa": 61.44,
    "Ugo Humbert": 60.36, "Matteo Berrettini": 60.00, "Benoit Paire": 55.29,
    "David Goffin": 55.05
})
df_2019['first_serve_won_pct'] = df_2019['player'].map({
    "Milos Raonic": 84.78, "Sam Querrey": 83.58, "Matteo Berrettini": 81.11,
    "Rafael Nadal": 80.84, "Roger Federer": 79.96, "David Goffin": 79.40,
    "Benoit Paire": 78.13, "Novak Djokovic": 78.00, "Fernando Verdasco": 77.05,
    "Roberto Bautista Agut": 75.33, "Tennys Sandgren": 73.86, "Guido Pella": 72.89,
    "Ugo Humbert": 72.08, "Kei Nishikori": 70.96, "Mikhail Kukushkin": 69.82,
    "Joao Sousa": 68.09
})
df_2019['second_serve_won_pct'] = df_2019['player'].map({
    "Milos Raonic": 62.32, "Roger Federer": 61.54, "Rafael Nadal": 60.68,
    "Kei Nishikori": 59.17, "Roberto Bautista Agut": 57.63, "Mikhail Kukushkin": 55.94,
    "Joao Sousa": 55.37, "Tennys Sandgren": 54.26, "Novak Djokovic": 52.75,
    "Ugo Humbert": 52.30, "Fernando Verdasco": 51.67, "Benoit Paire": 50.72,
    "Sam Querrey": 50.47, "David Goffin": 50.46, "Guido Pella": 50.23,
    "Matteo Berrettini": 50.00
})
df_2019['serve_pts_won_pct'] = df_2019['player'].map({
    "Milos Raonic": 77.29, "Roger Federer": 73.48, "Rafael Nadal": 73.15,
    "Sam Querrey": 70.89, "Roberto Bautista Agut": 69.68, "Novak Djokovic": 68.98,
    "Matteo Berrettini": 68.67, "Fernando Verdasco": 67.37, "Kei Nishikori": 66.74,
    "Tennys Sandgren": 66.73, "David Goffin": 66.39, "Mikhail Kukushkin": 66.03,
    "Guido Pella": 66.03, "Benoit Paire": 65.87, "Ugo Humbert": 64.24,
    "Joao Sousa": 63.18
})
df_2019['ace_pct'] = df_2019['player'].map({
    "Milos Raonic": 22.22, "Sam Querrey": 22.06, "Benoit Paire": 13.82,
    "Matteo Berrettini": 13.33, "Fernando Verdasco": 12.29, "Roger Federer": 10.96,
    "Rafael Nadal": 10.56, "Tennys Sandgren": 10.44, "David Goffin": 7.84,
    "Novak Djokovic": 7.72, "Joao Sousa": 6.54, "Ugo Humbert": 5.92,
    "Mikhail Kukushkin": 5.15, "Guido Pella": 4.79, "Roberto Bautista Agut": 4.33,
    "Kei Nishikori": 3.18
})
df_2019['double_fault_pct'] = df_2019['player'].map({
    "Benoit Paire": 6.91, "Fernando Verdasco": 5.51, "Milos Raonic": 4.59,
    "David Goffin": 4.33, "Ugo Humbert": 4.33, "Sam Querrey": 3.98,
    "Novak Djokovic": 3.80, "Matteo Berrettini": 3.78, "Rafael Nadal": 2.59,
    "Kei Nishikori": 2.54, "Guido Pella": 2.19, "Tennys Sandgren": 2.13,
    "Roger Federer": 2.03, "Roberto Bautista Agut": 1.99, "Joao Sousa": 0.87,
    "Mikhail Kukushkin": 0.38
})
df_2019['bp_saved_pct'] = df_2019['player'].map({
    "Milos Raonic": 80.95, "Mikhail Kukushkin": 79.49, "Sam Querrey": 78.13,
    "Guido Pella": 75.81, "Rafael Nadal": 75.00, "Roger Federer": 73.33,
    "Roberto Bautista Agut": 71.88, "Joao Sousa": 68.09, "Fernando Verdasco": 66.67,
    "Ugo Humbert": 65.71, "Benoit Paire": 65.52, "Novak Djokovic": 64.86,
    "Matteo Berrettini": 64.52, "Kei Nishikori": 62.07, "Tennys Sandgren": 56.67,
    "David Goffin": 44.83
})
df_2019['return_pts_won_pct'] = df_2019['player'].map({
    "Novak Djokovic": 40.97, "Roberto Bautista Agut": 40.95, "Kei Nishikori": 40.80,
    "Roger Federer": 40.34, "Rafael Nadal": 39.33, "David Goffin": 39.24,
    "Joao Sousa": 38.16, "Ugo Humbert": 37.16, "Fernando Verdasco": 37.01,
    "Mikhail Kukushkin": 36.56, "Milos Raonic": 36.55, "Tennys Sandgren": 36.50,
    "Benoit Paire": 33.33, "Guido Pella": 33.33, "Matteo Berrettini": 33.26,
    "Sam Querrey": 33.21
})
df_2019['second_serve_return_won_pct'] = df_2019['player'].map({
    "Roberto Bautista Agut": 58.63, "Kei Nishikori": 58.01, "Novak Djokovic": 54.31,
    "Rafael Nadal": 53.57, "Ugo Humbert": 53.38, "Fernando Verdasco": 52.63,
    "Joao Sousa": 51.87, "Roger Federer": 51.28, "Guido Pella": 51.03,
    "Tennys Sandgren": 50.83, "David Goffin": 50.66, "Benoit Paire": 49.61,
    "Mikhail Kukushkin": 48.90, "Milos Raonic": 46.21, "Matteo Berrettini": 46.07,
    "Sam Querrey": 42.38
})
df_2019['bp_won_pct'] = df_2019['player'].map({
    "Joao Sousa": 48.72, "Tennys Sandgren": 48.72, "Roger Federer": 47.14,
    "Novak Djokovic": 46.84, "Rafael Nadal": 42.86, "Kei Nishikori": 42.31,
    "Fernando Verdasco": 41.46, "Sam Querrey": 40.00, "Ugo Humbert": 38.46,
    "Roberto Bautista Agut": 36.51, "David Goffin": 34.48, "Mikhail Kukushkin": 32.56,
    "Milos Raonic": 31.58, "Matteo Berrettini": 31.25, "Guido Pella": 28.57,
    "Benoit Paire": 27.59
})
df_2019['tiebreak_won_pct'] = df_2019['player'].map({
    "Benoit Paire": 100.00, "Guido Pella": 100.00, "Matteo Berrettini": 80.00,
    "Milos Raonic": 80.00, "Novak Djokovic": 75.00, "Sam Querrey": 71.43,
    "Fernando Verdasco": 66.67, "Rafael Nadal": 66.67, "Mikhail Kukushkin": 50.00,
    "Roger Federer": 50.00, "Tennys Sandgren": 50.00, "Ugo Humbert": 50.00,
    "Roberto Bautista Agut": None, "Kei Nishikori": None, "David Goffin": None,
    "Joao Sousa": None
})
df_2019['points_dominance'] = df_2019['player'].map({
    "Milos Raonic": 1.610, "Roger Federer": 1.521, "Rafael Nadal": 1.465,
    "Roberto Bautista Agut": 1.351, "Novak Djokovic": 1.321, "Kei Nishikori": 1.227,
    "David Goffin": 1.167, "Sam Querrey": 1.141, "Fernando Verdasco": 1.134,
    "Tennys Sandgren": 1.097, "Mikhail Kukushkin": 1.076, "Matteo Berrettini": 1.062,
    "Ugo Humbert": 1.039, "Joao Sousa": 1.036, "Guido Pella": 0.981,
    "Benoit Paire": 0.977
})
df_2019['point_time'] = df_2019['player'].map({
    "Rafael Nadal": 44.82, "Novak Djokovic": 41.98, "Fernando Verdasco": 41.69,
    "Guido Pella": 41.09, "Joao Sousa": 40.87, "David Goffin": 40.60,
    "Kei Nishikori": 40.06, "Ugo Humbert": 40.02, "Matteo Berrettini": 39.89,
    "Roger Federer": 39.81, "Tennys Sandgren": 39.77, "Roberto Bautista Agut": 39.45,
    "Milos Raonic": 39.44, "Mikhail Kukushkin": 38.68, "Sam Querrey": 35.78,
    "Benoit Paire": 35.76
})
df_2019['round_reached'] = df_2019['player'].map({
    "Novak Djokovic": "W", "Roger Federer": "F", "Roberto Bautista Agut": "SF",
    "Rafael Nadal": "SF", "Milos Raonic": "R16", "Kei Nishikori": "QF",
    "Guido Pella": "QF", "David Goffin": "QF", "Mikhail Kukushkin": "R16",
    "Tennys Sandgren": "R16", "Fernando Verdasco": "R16", "Sam Querrey": "QF",
    "Joao Sousa": "R16", "Ugo Humbert": "R16", "Matteo Berrettini": "R16",
    "Benoit Paire": "R16"
})
df_2019['year'] = 2019

grass_elo_2019 = {
    "Novak Djokovic": 2379, "Roger Federer": 2364, "Rafael Nadal": 2412,
    "Roberto Bautista Agut": 2082, "Milos Raonic": 2102, "Guido Pella": 1947,
    "David Goffin": 2071, "Mikhail Kukushkin": 1859, "Tennys Sandgren": 1791,
    "Fernando Verdasco": 1985, "Sam Querrey": 1951, "Joao Sousa": 1855,
    "Ugo Humbert": 1768, "Matteo Berrettini": 2019, "Benoit Paire": 1991,
    "Kei Nishikori": 2152
}
df_2019['grass_elo'] = df_2019['player'].map(grass_elo_2019)

pre_wimbledon_form_2019 = {
    "Novak Djokovic": 0, "Roger Federer": 5, "Rafael Nadal": 0,
    "Roberto Bautista Agut": 2, "Milos Raonic": 2, "Kei Nishikori": 0,
    "Guido Pella": 1, "David Goffin": 4, "Mikhail Kukushkin": 1,
    "Tennys Sandgren": 0, "Fernando Verdasco": 1, "Sam Querrey": 0,
    "Joao Sousa": 1, "Ugo Humbert": 0, "Matteo Berrettini": 3, "Benoit Paire": 1
}
df_2019['pre_wimbledon_form'] = df_2019['player'].map(pre_wimbledon_form_2019)

print(f"✅ 2019 data loaded: {df_2019.shape}")

In [ ]:
# SECTION 3 — HISTORICAL DATA: 2021
# ===============================================================================
players_2021 = [
    "Alexander Zverev", "Lorenzo Sonego", "Sebastian Korda", "Karen Khachanov",
    "Roberto Bautista Agut", "Roger Federer", "Christian Garin", "Novak Djokovic",
    "Felix Auger Aliassime", "Hubert Hurkacz", "Andrey Rublev", "Matteo Berrettini",
    "Denis Shapovalov", "Ilya Ivashka", "Marton Fucsovics", "Daniil Medvedev"
]
df_2021 = pd.DataFrame({'player': players_2021})

df_2021['first_serve_pct'] = df_2021['player'].map({
    "Alexander Zverev": 68.00, "Lorenzo Sonego": 67.70, "Sebastian Korda": 65.38,
    "Karen Khachanov": 65.02, "Roberto Bautista Agut": 64.60, "Roger Federer": 63.62,
    "Christian Garin": 62.93, "Novak Djokovic": 62.91, "Felix Auger Aliassime": 62.00,
    "Hubert Hurkacz": 61.95, "Andrey Rublev": 61.47, "Matteo Berrettini": 60.03,
    "Denis Shapovalov": 59.53, "Ilya Ivashka": 58.65, "Marton Fucsovics": 58.64,
    "Daniil Medvedev": 58.61
})
df_2021['first_serve_won_pct'] = df_2021['player'].map({
    "Novak Djokovic": 84.08, "Alexander Zverev": 83.04, "Daniil Medvedev": 81.00,
    "Matteo Berrettini": 80.45, "Denis Shapovalov": 78.75, "Hubert Hurkacz": 78.26,
    "Ilya Ivashka": 76.50, "Felix Auger Aliassime": 76.09, "Lorenzo Sonego": 75.95,
    "Roger Federer": 75.08, "Marton Fucsovics": 74.29, "Andrey Rublev": 73.51,
    "Karen Khachanov": 71.88, "Christian Garin": 71.66, "Sebastian Korda": 70.29,
    "Roberto Bautista Agut": 65.41
})
df_2021['second_serve_won_pct'] = df_2021['player'].map({
    "Felix Auger Aliassime": 58.24, "Roger Federer": 56.98, "Matteo Berrettini": 56.31,
    "Hubert Hurkacz": 56.19, "Novak Djokovic": 56.12, "Sebastian Korda": 55.00,
    "Karen Khachanov": 53.64, "Andrey Rublev": 53.57, "Ilya Ivashka": 50.98,
    "Roberto Bautista Agut": 50.63, "Christian Garin": 50.27, "Daniil Medvedev": 49.75,
    "Denis Shapovalov": 49.17, "Alexander Zverev": 48.53, "Lorenzo Sonego": 48.00,
    "Marton Fucsovics": 47.56
})
df_2021['serve_pts_won_pct'] = df_2021['player'].map({
    "Novak Djokovic": 73.71, "Alexander Zverev": 72.00, "Matteo Berrettini": 70.80,
    "Hubert Hurkacz": 69.87, "Felix Auger Aliassime": 69.31, "Roger Federer": 68.50,
    "Daniil Medvedev": 68.07, "Lorenzo Sonego": 66.93, "Denis Shapovalov": 66.78,
    "Ilya Ivashka": 65.95, "Andrey Rublev": 65.83, "Karen Khachanov": 65.50,
    "Sebastian Korda": 65.00, "Christian Garin": 63.73, "Marton Fucsovics": 63.24,
    "Roberto Bautista Agut": 60.18
})
df_2021['ace_pct'] = df_2021['player'].map({
    "Matteo Berrettini": 15.96, "Alexander Zverev": 14.35, "Felix Auger Aliassime": 12.94,
    "Lorenzo Sonego": 12.40, "Denis Shapovalov": 10.96, "Ilya Ivashka": 10.81,
    "Daniil Medvedev": 10.71, "Novak Djokovic": 10.64, "Hubert Hurkacz": 9.09,
    "Karen Khachanov": 8.90, "Christian Garin": 8.62, "Roger Federer": 8.54,
    "Andrey Rublev": 7.80, "Sebastian Korda": 6.54, "Marton Fucsovics": 5.51,
    "Roberto Bautista Agut": 3.54
})
df_2021['double_fault_pct'] = df_2021['player'].map({
    "Alexander Zverev": 8.24, "Denis Shapovalov": 7.25, "Marton Fucsovics": 5.51,
    "Lorenzo Sonego": 3.62, "Felix Auger Aliassime": 3.55, "Novak Djokovic": 3.29,
    "Ilya Ivashka": 2.97, "Roberto Bautista Agut": 2.65, "Karen Khachanov": 2.54,
    "Daniil Medvedev": 2.52, "Christian Garin": 2.40, "Sebastian Korda": 2.31,
    "Andrey Rublev": 2.29, "Matteo Berrettini": 2.05, "Hubert Hurkacz": 1.68,
    "Roger Federer": 1.42
})
df_2021['bp_saved_pct'] = df_2021['player'].map({
    "Novak Djokovic": 78.79, "Denis Shapovalov": 76.92, "Matteo Berrettini": 73.81,
    "Roger Federer": 71.43, "Karen Khachanov": 70.18, "Christian Garin": 66.67,
    "Hubert Hurkacz": 61.54, "Lorenzo Sonego": 61.54, "Alexander Zverev": 60.00,
    "Marton Fucsovics": 59.52, "Andrey Rublev": 57.14, "Sebastian Korda": 56.10,
    "Felix Auger Aliassime": 51.43, "Roberto Bautista Agut": 48.65, "Ilya Ivashka": 47.62,
    "Daniil Medvedev": 45.00
})
df_2021['return_pts_won_pct'] = df_2021['player'].map({
    "Novak Djokovic": 41.27, "Roberto Bautista Agut": 41.18, "Sebastian Korda": 40.55,
    "Marton Fucsovics": 40.23, "Andrey Rublev": 40.14, "Daniil Medvedev": 39.70,
    "Lorenzo Sonego": 39.18, "Denis Shapovalov": 38.97, "Alexander Zverev": 38.35,
    "Ilya Ivashka": 38.13, "Roger Federer": 37.91, "Felix Auger Aliassime": 37.72,
    "Matteo Berrettini": 37.48, "Christian Garin": 36.88, "Karen Khachanov": 36.66,
    "Hubert Hurkacz": 36.02
})
df_2021['second_serve_return_won_pct'] = df_2021['player'].map({
    "Daniil Medvedev": 59.90, "Roberto Bautista Agut": 58.76, "Marton Fucsovics": 57.51,
    "Andrey Rublev": 56.98, "Novak Djokovic": 55.48, "Felix Auger Aliassime": 53.85,
    "Sebastian Korda": 52.79, "Hubert Hurkacz": 52.72, "Ilya Ivashka": 52.17,
    "Karen Khachanov": 51.96, "Roger Federer": 51.90, "Lorenzo Sonego": 50.76,
    "Alexander Zverev": 49.18, "Denis Shapovalov": 49.12, "Christian Garin": 48.50,
    "Matteo Berrettini": 48.31
})
df_2021['bp_won_pct'] = df_2021['player'].map({
    "Karen Khachanov": 54.35, "Lorenzo Sonego": 51.52, "Ilya Ivashka": 48.48,
    "Andrey Rublev": 47.62, "Daniil Medvedev": 46.81, "Roberto Bautista Agut": 45.83,
    "Marton Fucsovics": 45.76, "Matteo Berrettini": 44.78, "Alexander Zverev": 44.68,
    "Hubert Hurkacz": 41.67, "Felix Auger Aliassime": 38.78, "Novak Djokovic": 38.10,
    "Sebastian Korda": 35.29, "Denis Shapovalov": 34.85, "Roger Federer": 34.69,
    "Christian Garin": 25.42
})
df_2021['tiebreak_won_pct'] = df_2021['player'].map({
    "Felix Auger Aliassime": 100.00, "Hubert Hurkacz": 100.00, "Karen Khachanov": 100.00,
    "Christian Garin": 66.67, "Matteo Berrettini": 66.67, "Novak Djokovic": 66.67,
    "Roberto Bautista Agut": 66.67, "Sebastian Korda": 50.00, "Alexander Zverev": 33.33,
    "Daniil Medvedev": 33.33, "Roger Federer": 33.33, "Denis Shapovalov": None,
    "Andrey Rublev": None, "Lorenzo Sonego": None, "Ilya Ivashka": None,
    "Marton Fucsovics": None
})
df_2021['points_dominance'] = df_2021['player'].map({
    "Novak Djokovic": 1.570, "Alexander Zverev": 1.370, "Matteo Berrettini": 1.284,
    "Daniil Medvedev": 1.243, "Felix Auger Aliassime": 1.229, "Roger Federer": 1.203,
    "Hubert Hurkacz": 1.195, "Lorenzo Sonego": 1.185, "Andrey Rublev": 1.174,
    "Denis Shapovalov": 1.173, "Sebastian Korda": 1.159, "Ilya Ivashka": 1.120,
    "Marton Fucsovics": 1.094, "Karen Khachanov": 1.063, "Roberto Bautista Agut": 1.034,
    "Christian Garin": 1.017
})
df_2021['point_time'] = df_2021['player'].map({
    "Felix Auger Aliassime": 44.32, "Marton Fucsovics": 43.64, "Denis Shapovalov": 43.24,
    "Roberto Bautista Agut": 42.87, "Novak Djokovic": 42.69, "Ilya Ivashka": 41.80,
    "Matteo Berrettini": 41.74, "Lorenzo Sonego": 40.69, "Sebastian Korda": 40.62,
    "Karen Khachanov": 40.25, "Alexander Zverev": 39.73, "Roger Federer": 38.43,
    "Daniil Medvedev": 38.39, "Andrey Rublev": 38.11, "Hubert Hurkacz": 37.19,
    "Christian Garin": 28.33
})
df_2021['round_reached'] = df_2021['player'].map({
    "Novak Djokovic": "W", "Matteo Berrettini": "F", "Hubert Hurkacz": "SF",
    "Denis Shapovalov": "SF", "Roger Federer": "QF", "Alexander Zverev": "R16",
    "Felix Auger Aliassime": "QF", "Karen Khachanov": "QF", "Lorenzo Sonego": "R16",
    "Sebastian Korda": "R16", "Roberto Bautista Agut": "R16", "Christian Garin": "R16",
    "Andrey Rublev": "R16", "Ilya Ivashka": "R16", "Marton Fucsovics": "QF",
    "Daniil Medvedev": "R16"
})
df_2021['year'] = 2021

grass_elo_2021 = {
    "Novak Djokovic": 2418, "Matteo Berrettini": 2144, "Hubert Hurkacz": 1933,
    "Denis Shapovalov": 2003, "Roger Federer": 2270, "Alexander Zverev": 2196,
    "Felix Auger Aliassime": 1984, "Karen Khachanov": 1956, "Lorenzo Sonego": 1966,
    "Sebastian Korda": 1934, "Roberto Bautista Agut": 2028, "Christian Garin": 1996,
    "Andrey Rublev": 2166, "Ilya Ivashka": 1843, "Marton Fucsovics": 1959,
    "Daniil Medvedev": 2241
}
df_2021['grass_elo'] = df_2021['player'].map(grass_elo_2021)

pre_wimbledon_form_2021 = {
    "Novak Djokovic": 0, "Matteo Berrettini": 5, "Hubert Hurkacz": 1,
    "Denis Shapovalov": 3, "Roger Federer": 1, "Alexander Zverev": 1,
    "Felix Auger Aliassime": 3, "Karen Khachanov": 1, "Lorenzo Sonego": 1,
    "Sebastian Korda": 2, "Roberto Bautista Agut": 1, "Christian Garin": 1,
    "Andrey Rublev": 4, "Ilya Ivashka": 1, "Marton Fucsovics": 0,
    "Daniil Medvedev": 1
}
df_2021['pre_wimbledon_form'] = df_2021['player'].map(pre_wimbledon_form_2021)

print(f"✅ 2021 data loaded: {df_2021.shape}")

In [ ]:
# SECTION 4 — HISTORICAL DATA: 2022
# ===============================================================================
players_2022 = [
    "Nick Kyrgios", "Taylor Harry Fritz", "Rafael Nadal", "Tommy Paul",
    "Tim Van Rijthoven", "Cameron Norrie", "Carlos Alcaraz Garfia", "Jason Kubler",
    "Brandon Nakashima", "Novak Djokovic", "Christian Garin", "David Goffin",
    "Botic Van De Zandschulp", "Jannik Sinner", "Frances Tiafoe", "Alex De Minaur"
]
df_2022 = pd.DataFrame({'player': players_2022})

df_2022['first_serve_pct'] = df_2022['player'].map({
    "Nick Kyrgios": 70.50, "Taylor Harry Fritz": 67.00, "Rafael Nadal": 66.19,
    "Tommy Paul": 65.50, "Tim Van Rijthoven": 64.58, "Cameron Norrie": 64.24,
    "Carlos Alcaraz Garfia": 63.96, "Jason Kubler": 63.73, "Brandon Nakashima": 63.57,
    "Novak Djokovic": 63.02, "Christian Garin": 61.55, "David Goffin": 60.82,
    "Botic Van De Zandschulp": 59.02, "Jannik Sinner": 57.91, "Frances Tiafoe": 56.29,
    "Alex De Minaur": 54.94
})
df_2022['first_serve_won_pct'] = df_2022['player'].map({
    "Tim Van Rijthoven": 82.84, "Novak Djokovic": 82.39, "Frances Tiafoe": 81.32,
    "Botic Van De Zandschulp": 80.79, "Brandon Nakashima": 80.78, "Taylor Harry Fritz": 79.35,
    "Nick Kyrgios": 76.97, "Jannik Sinner": 75.87, "Carlos Alcaraz Garfia": 75.60,
    "Jason Kubler": 74.62, "David Goffin": 73.68, "Alex De Minaur": 73.03,
    "Christian Garin": 72.62, "Tommy Paul": 72.32, "Rafael Nadal": 71.93,
    "Cameron Norrie": 70.75
})
df_2022['second_serve_won_pct'] = df_2022['player'].map({
    "Tommy Paul": 60.17, "Brandon Nakashima": 59.63, "Jannik Sinner": 59.60,
    "Carlos Alcaraz Garfia": 58.54, "Novak Djokovic": 56.80, "Alex De Minaur": 56.16,
    "Frances Tiafoe": 54.72, "Rafael Nadal": 54.45, "Cameron Norrie": 54.24,
    "David Goffin": 52.92, "Nick Kyrgios": 52.75, "Jason Kubler": 52.03,
    "Botic Van De Zandschulp": 51.57, "Tim Van Rijthoven": 51.02, "Christian Garin": 50.25,
    "Taylor Harry Fritz": 49.70
})
df_2022['serve_pts_won_pct'] = df_2022['player'].map({
    "Brandon Nakashima": 73.08, "Novak Djokovic": 72.93, "Tim Van Rijthoven": 71.57,
    "Nick Kyrgios": 69.82, "Frances Tiafoe": 69.69, "Taylor Harry Fritz": 69.57,
    "Carlos Alcaraz Garfia": 69.45, "Jannik Sinner": 69.02, "Botic Van De Zandschulp": 68.81,
    "Tommy Paul": 68.13, "Jason Kubler": 66.42, "Rafael Nadal": 66.02,
    "David Goffin": 65.55, "Alex De Minaur": 65.43, "Cameron Norrie": 64.85,
    "Christian Garin": 64.02
})
df_2022['ace_pct'] = df_2022['player'].map({
    "Nick Kyrgios": 20.30, "Tim Van Rijthoven": 17.59, "Taylor Harry Fritz": 15.22,
    "Botic Van De Zandschulp": 12.89, "Carlos Alcaraz Garfia": 11.21, "Novak Djokovic": 9.62,
    "Frances Tiafoe": 9.07, "Alex De Minaur": 8.02, "Jason Kubler": 7.35,
    "David Goffin": 7.01, "Brandon Nakashima": 6.33, "Rafael Nadal": 6.02,
    "Cameron Norrie": 5.30, "Tommy Paul": 4.68, "Jannik Sinner": 4.55,
    "Christian Garin": 3.79
})
df_2022['double_fault_pct'] = df_2022['player'].map({
    "Botic Van De Zandschulp": 6.70, "Tim Van Rijthoven": 5.06, "Christian Garin": 4.36,
    "David Goffin": 4.27, "Jason Kubler": 4.17, "Jannik Sinner": 3.87,
    "Carlos Alcaraz Garfia": 3.74, "Alex De Minaur": 3.70, "Nick Kyrgios": 3.65,
    "Novak Djokovic": 3.55, "Rafael Nadal": 3.54, "Frances Tiafoe": 2.89,
    "Taylor Harry Fritz": 2.57, "Brandon Nakashima": 2.26, "Tommy Paul": 2.05,
    "Cameron Norrie": 1.52
})
df_2022['bp_saved_pct'] = df_2022['player'].map({
    "Nick Kyrgios": 78.95, "Carlos Alcaraz Garfia": 74.07, "Tommy Paul": 71.43,
    "Alex De Minaur": 70.00, "Tim Van Rijthoven": 68.97, "Jannik Sinner": 66.67,
    "Taylor Harry Fritz": 65.71, "Frances Tiafoe": 65.52, "Cameron Norrie": 65.31,
    "Novak Djokovic": 64.71, "David Goffin": 63.27, "Brandon Nakashima": 61.54,
    "Jason Kubler": 60.00, "Rafael Nadal": 59.52, "Botic Van De Zandschulp": 56.52,
    "Christian Garin": 52.78
})
df_2022['return_pts_won_pct'] = df_2022['player'].map({
    "Alex De Minaur": 42.63, "Christian Garin": 42.51, "Tommy Paul": 42.05,
    "Novak Djokovic": 41.54, "Taylor Harry Fritz": 41.09, "Rafael Nadal": 40.87,
    "David Goffin": 40.68, "Cameron Norrie": 39.64, "Botic Van De Zandschulp": 38.48,
    "Jannik Sinner": 37.74, "Frances Tiafoe": 37.52, "Carlos Alcaraz Garfia": 36.42,
    "Jason Kubler": 35.50, "Brandon Nakashima": 34.59, "Nick Kyrgios": 34.26,
    "Tim Van Rijthoven": 33.10
})
df_2022['second_serve_return_won_pct'] = df_2022['player'].map({
    "Taylor Harry Fritz": 59.77, "Novak Djokovic": 57.24, "Frances Tiafoe": 56.31,
    "Rafael Nadal": 53.30, "Alex De Minaur": 53.01, "David Goffin": 52.17,
    "Cameron Norrie": 52.12, "Jason Kubler": 52.05, "Christian Garin": 51.72,
    "Tommy Paul": 51.52, "Botic Van De Zandschulp": 51.27, "Jannik Sinner": 50.00,
    "Carlos Alcaraz Garfia": 48.51, "Nick Kyrgios": 47.37, "Brandon Nakashima": 47.16,
    "Tim Van Rijthoven": 40.76
})
df_2022['bp_won_pct'] = df_2022['player'].map({
    "Alex De Minaur": 52.08, "Taylor Harry Fritz": 51.92, "Cameron Norrie": 49.25,
    "Rafael Nadal": 44.44, "David Goffin": 44.12, "Botic Van De Zandschulp": 43.24,
    "Jason Kubler": 42.86, "Tommy Paul": 42.50, "Novak Djokovic": 40.00,
    "Christian Garin": 39.68, "Frances Tiafoe": 39.53, "Tim Van Rijthoven": 39.39,
    "Brandon Nakashima": 37.21, "Nick Kyrgios": 37.04, "Jannik Sinner": 35.59,
    "Carlos Alcaraz Garfia": 22.00
})
df_2022['tiebreak_won_pct'] = df_2022['player'].map({
    "Carlos Alcaraz Garfia": 100.00, "David Goffin": 100.00, "Rafael Nadal": 100.00,
    "Frances Tiafoe": 75.00, "Tim Van Rijthoven": 75.00, "Christian Garin": 66.67,
    "Taylor Harry Fritz": 66.67, "Botic Van De Zandschulp": 50.00, "Jannik Sinner": 50.00,
    "Nick Kyrgios": 50.00, "Alex De Minaur": 33.33, "Brandon Nakashima": 33.33,
    "Novak Djokovic": None, "Tommy Paul": None, "Cameron Norrie": None,
    "Jason Kubler": None
})
df_2022['points_dominance'] = df_2022['player'].map({
    "Novak Djokovic": 1.534, "Taylor Harry Fritz": 1.350, "Tommy Paul": 1.319,
    "Brandon Nakashima": 1.285, "Frances Tiafoe": 1.238, "Botic Van De Zandschulp": 1.234,
    "Alex De Minaur": 1.233, "Jannik Sinner": 1.218, "Rafael Nadal": 1.203,
    "Carlos Alcaraz Garfia": 1.192, "Christian Garin": 1.181, "David Goffin": 1.181,
    "Tim Van Rijthoven": 1.164, "Nick Kyrgios": 1.135, "Cameron Norrie": 1.128,
    "Jason Kubler": 1.057
})
df_2022['point_time'] = df_2022['player'].map({
    "Rafael Nadal": 47.61, "Jason Kubler": 46.11, "Jannik Sinner": 45.17,
    "Novak Djokovic": 44.84, "Alex De Minaur": 43.30, "Christian Garin": 43.23,
    "Carlos Alcaraz Garfia": 42.03, "Cameron Norrie": 41.99, "Taylor Harry Fritz": 41.62,
    "Botic Van De Zandschulp": 40.33, "David Goffin": 40.00, "Brandon Nakashima": 39.71,
    "Nick Kyrgios": 39.01, "Tim Van Rijthoven": 38.95, "Frances Tiafoe": 38.36,
    "Tommy Paul": 38.20
})
df_2022['round_reached'] = df_2022['player'].map({
    "Novak Djokovic": "W", "Nick Kyrgios": "F", "Cameron Norrie": "SF",
    "Rafael Nadal": "SF", "Taylor Harry Fritz": "QF", "David Goffin": "QF",
    "Botic Van De Zandschulp": "QF", "Carlos Alcaraz Garfia": "R16", "Jannik Sinner": "R16",
    "Tommy Paul": "R16", "Tim Van Rijthoven": "R16", "Jason Kubler": "R16",
    "Brandon Nakashima": "R16", "Christian Garin": "QF", "Frances Tiafoe": "R16",
    "Alex De Minaur": "R16"
})
df_2022['year'] = 2022

grass_elo_2022 = {
    "Novak Djokovic": 2367, "Nick Kyrgios": 2082, "Cameron Norrie": 2046,
    "Rafael Nadal": 2401, "Taylor Harry Fritz": 2053, "David Goffin": 1938,
    "Botic Van De Zandschulp": 1949, "Carlos Alcaraz Garfia": 2207, "Jannik Sinner": 2117,
    "Tommy Paul": 1957, "Tim Van Rijthoven": 1870, "Jason Kubler": 1806,
    "Brandon Nakashima": 1894, "Christian Garin": 1930, "Frances Tiafoe": 1933,
    "Alex De Minaur": 1993
}
df_2022['grass_elo'] = df_2022['player'].map(grass_elo_2022)

pre_wimbledon_form_2022 = {
    "Novak Djokovic": 0, "Nick Kyrgios": 3, "Cameron Norrie": 1,
    "Rafael Nadal": 0, "Taylor Harry Fritz": 1, "David Goffin": 1,
    "Botic Van De Zandschulp": 3, "Carlos Alcaraz Garfia": 0, "Jannik Sinner": 0,
    "Tommy Paul": 2, "Tim Van Rijthoven": 0, "Jason Kubler": 0,
    "Brandon Nakashima": 0, "Christian Garin": 0, "Frances Tiafoe": 1,
    "Alex De Minaur": 1
}
df_2022['pre_wimbledon_form'] = df_2022['player'].map(pre_wimbledon_form_2022)

print(f"✅ 2022 data loaded: {df_2022.shape}")

In [ ]:
# SECTION 5 — HISTORICAL DATA: 2023
# ===============================================================================
players_2023 = [
    "Christopher Eubanks", "Hubert Hurkacz", "Stefanos Tsitsipas",
    "Daniil Medvedev", "Jiri Lehecka", "Matteo Berrettini",
    "Novak Djokovic", "Roman Safiullin", "Alexander Bublik",
    "Daniel Elahi Galan Riveros", "Carlos Alcaraz Garfia", "Andrey Rublev",
    "Holger Vitus Nodskov Rune", "Denis Shapovalov", "Grigor Dimitrov",
    "Jannik Sinner"
]
df_2023 = pd.DataFrame({'player': players_2023})

df_2023['first_serve_pct'] = df_2023['player'].map({
    "Christopher Eubanks": 70.34, "Hubert Hurkacz": 69.19, "Stefanos Tsitsipas": 67.62,
    "Daniil Medvedev": 67.32, "Jiri Lehecka": 67.25, "Matteo Berrettini": 66.30,
    "Novak Djokovic": 64.61, "Roman Safiullin": 64.59, "Alexander Bublik": 64.33,
    "Daniel Elahi Galan Riveros": 63.89, "Carlos Alcaraz Garfia": 63.42, "Andrey Rublev": 62.79,
    "Holger Vitus Nodskov Rune": 62.10, "Denis Shapovalov": 61.26, "Grigor Dimitrov": 58.22,
    "Jannik Sinner": 56.44
})
df_2023['first_serve_won_pct'] = df_2023['player'].map({
    "Alexander Bublik": 87.02, "Grigor Dimitrov": 83.33, "Jannik Sinner": 82.81,
    "Hubert Hurkacz": 81.98, "Stefanos Tsitsipas": 80.79, "Matteo Berrettini": 80.40,
    "Christopher Eubanks": 78.37, "Daniil Medvedev": 78.10, "Denis Shapovalov": 77.94,
    "Holger Vitus Nodskov Rune": 77.66, "Novak Djokovic": 77.58, "Andrey Rublev": 77.50,
    "Carlos Alcaraz Garfia": 76.35, "Jiri Lehecka": 75.66, "Roman Safiullin": 73.28,
    "Daniel Elahi Galan Riveros": 72.05
})
df_2023['second_serve_won_pct'] = df_2023['player'].map({
    "Jannik Sinner": 62.73, "Stefanos Tsitsipas": 60.99, "Novak Djokovic": 60.85,
    "Jiri Lehecka": 59.23, "Roman Safiullin": 57.79, "Carlos Alcaraz Garfia": 57.19,
    "Hubert Hurkacz": 57.14, "Andrey Rublev": 54.43, "Matteo Berrettini": 54.25,
    "Grigor Dimitrov": 54.19, "Holger Vitus Nodskov Rune": 53.57, "Christopher Eubanks": 51.83,
    "Alexander Bublik": 50.00, "Denis Shapovalov": 47.67, "Daniel Elahi Galan Riveros": 47.25,
    "Daniil Medvedev": 46.74
})
df_2023['serve_pts_won_pct'] = df_2023['player'].map({
    "Stefanos Tsitsipas": 74.38, "Hubert Hurkacz": 74.33, "Jannik Sinner": 74.06,
    "Alexander Bublik": 73.81, "Novak Djokovic": 71.66, "Matteo Berrettini": 71.59,
    "Grigor Dimitrov": 71.16, "Christopher Eubanks": 70.50, "Jiri Lehecka": 70.28,
    "Carlos Alcaraz Garfia": 69.34, "Andrey Rublev": 68.92, "Holger Vitus Nodskov Rune": 68.53,
    "Daniil Medvedev": 67.85, "Roman Safiullin": 67.79, "Denis Shapovalov": 66.22,
    "Daniel Elahi Galan Riveros": 63.10
})
df_2023['ace_pct'] = df_2023['player'].map({
    "Hubert Hurkacz": 20.05, "Alexander Bublik": 18.96, "Christopher Eubanks": 15.84,
    "Jannik Sinner": 12.87, "Grigor Dimitrov": 12.40, "Denis Shapovalov": 11.94,
    "Matteo Berrettini": 11.89, "Daniil Medvedev": 11.01, "Andrey Rublev": 9.26,
    "Daniel Elahi Galan Riveros": 8.73, "Jiri Lehecka": 8.56, "Novak Djokovic": 8.56,
    "Roman Safiullin": 8.36, "Stefanos Tsitsipas": 8.36, "Holger Vitus Nodskov Rune": 7.95,
    "Carlos Alcaraz Garfia": 6.32
})
df_2023['double_fault_pct'] = df_2023['player'].map({
    "Alexander Bublik": 8.58, "Denis Shapovalov": 7.88, "Grigor Dimitrov": 5.39,
    "Holger Vitus Nodskov Rune": 5.08, "Roman Safiullin": 4.98, "Daniil Medvedev": 4.09,
    "Christopher Eubanks": 4.04, "Carlos Alcaraz Garfia": 3.95, "Jannik Sinner": 2.97,
    "Jiri Lehecka": 2.27, "Hubert Hurkacz": 2.20, "Daniel Elahi Galan Riveros": 1.98,
    "Matteo Berrettini": 1.98, "Stefanos Tsitsipas": 1.60, "Andrey Rublev": 1.41,
    "Novak Djokovic": 1.39
})
df_2023['bp_saved_pct'] = df_2023['player'].map({
    "Hubert Hurkacz": 94.74, "Jiri Lehecka": 82.61, "Matteo Berrettini": 81.82,
    "Grigor Dimitrov": 78.95, "Novak Djokovic": 78.95, "Jannik Sinner": 73.33,
    "Daniel Elahi Galan Riveros": 73.08, "Carlos Alcaraz Garfia": 72.50, "Alexander Bublik": 72.22,
    "Holger Vitus Nodskov Rune": 71.05, "Andrey Rublev": 65.71, "Denis Shapovalov": 61.76,
    "Daniil Medvedev": 61.11, "Stefanos Tsitsipas": 60.00, "Roman Safiullin": 55.56,
    "Christopher Eubanks": 51.85
})
df_2023['return_pts_won_pct'] = df_2023['player'].map({
    "Carlos Alcaraz Garfia": 40.69, "Grigor Dimitrov": 40.59, "Jannik Sinner": 40.51,
    "Daniil Medvedev": 39.21, "Denis Shapovalov": 38.50, "Roman Safiullin": 38.24,
    "Daniel Elahi Galan Riveros": 37.47, "Jiri Lehecka": 36.00, "Novak Djokovic": 35.49,
    "Alexander Bublik": 35.06, "Andrey Rublev": 34.55, "Holger Vitus Nodskov Rune": 33.83,
    "Hubert Hurkacz": 32.79, "Christopher Eubanks": 29.89, "Stefanos Tsitsipas": 29.32,
    "Matteo Berrettini": 28.29
})
df_2023['second_serve_return_won_pct'] = df_2023['player'].map({
    "Roman Safiullin": 52.50, "Daniil Medvedev": 52.42, "Jannik Sinner": 52.24,
    "Carlos Alcaraz Garfia": 51.93, "Grigor Dimitrov": 51.57, "Novak Djokovic": 51.17,
    "Daniel Elahi Galan Riveros": 50.58, "Jiri Lehecka": 49.22, "Alexander Bublik": 48.31,
    "Denis Shapovalov": 47.85, "Andrey Rublev": 47.66, "Hubert Hurkacz": 46.40,
    "Holger Vitus Nodskov Rune": 44.64, "Christopher Eubanks": 42.99, "Matteo Berrettini": 42.64,
    "Stefanos Tsitsipas": 39.55
})
df_2023['bp_won_pct'] = df_2023['player'].map({
    "Grigor Dimitrov": 59.38, "Daniel Elahi Galan Riveros": 55.56, "Jiri Lehecka": 46.67,
    "Denis Shapovalov": 46.15, "Roman Safiullin": 44.00, "Andrey Rublev": 42.55,
    "Daniil Medvedev": 41.07, "Alexander Bublik": 38.24, "Christopher Eubanks": 37.84,
    "Jannik Sinner": 36.11, "Hubert Hurkacz": 34.62, "Novak Djokovic": 33.33,
    "Carlos Alcaraz Garfia": 32.98, "Stefanos Tsitsipas": 32.35, "Matteo Berrettini": 31.82,
    "Holger Vitus Nodskov Rune": 31.11
})
df_2023['tiebreak_won_pct'] = df_2023['player'].map({
    "Daniil Medvedev": 100.00, "Novak Djokovic": 85.71, "Christopher Eubanks": 83.33,
    "Holger Vitus Nodskov Rune": 83.33, "Alexander Bublik": 80.00, "Carlos Alcaraz Garfia": 75.00,
    "Matteo Berrettini": 75.00, "Roman Safiullin": 75.00, "Stefanos Tsitsipas": 62.50,
    "Daniel Elahi Galan Riveros": 50.00, "Hubert Hurkacz": 50.00, "Jannik Sinner": 50.00,
    "Jiri Lehecka": 33.33, "Andrey Rublev": 20.00, "Grigor Dimitrov": 0.00,
    "Denis Shapovalov": None
})
df_2023['points_dominance'] = df_2023['player'].map({
    "Jannik Sinner": 1.562, "Grigor Dimitrov": 1.407, "Alexander Bublik": 1.339,
    "Carlos Alcaraz Garfia": 1.327, "Hubert Hurkacz": 1.277, "Novak Djokovic": 1.252,
    "Daniil Medvedev": 1.220, "Jiri Lehecka": 1.211, "Roman Safiullin": 1.187,
    "Stefanos Tsitsipas": 1.144, "Denis Shapovalov": 1.139, "Andrey Rublev": 1.112,
    "Holger Vitus Nodskov Rune": 1.075, "Daniel Elahi Galan Riveros": 1.015, "Christopher Eubanks": 1.013,
    "Matteo Berrettini": 0.996
})
df_2023['point_time'] = df_2023['player'].map({
    "Matteo Berrettini": 46.11, "Novak Djokovic": 45.94, "Carlos Alcaraz Garfia": 45.70,
    "Holger Vitus Nodskov Rune": 43.56, "Daniel Elahi Galan Riveros": 43.32, "Stefanos Tsitsipas": 43.28,
    "Jannik Sinner": 43.20, "Grigor Dimitrov": 42.23, "Daniil Medvedev": 42.02,
    "Roman Safiullin": 41.32, "Christopher Eubanks": 40.43, "Denis Shapovalov": 39.24,
    "Jiri Lehecka": 39.15, "Andrey Rublev": 38.42, "Hubert Hurkacz": 38.40,
    "Alexander Bublik": 36.95
})
df_2023['round_reached'] = df_2023['player'].map({
    "Carlos Alcaraz Garfia": "W", "Novak Djokovic": "F", "Daniil Medvedev": "SF",
    "Jannik Sinner": "SF", "Christopher Eubanks": "QF", "Roman Safiullin": "QF",
    "Andrey Rublev": "QF", "Holger Vitus Nodskov Rune": "QF", "Hubert Hurkacz": "R16",
    "Stefanos Tsitsipas": "R16", "Jiri Lehecka": "R16", "Matteo Berrettini": "R16",
    "Alexander Bublik": "R16", "Daniel Elahi Galan Riveros": "R16", "Denis Shapovalov": "R16",
    "Grigor Dimitrov": "R16"
})
df_2023['year'] = 2023

grass_elo_2023 = {
    "Carlos Alcaraz Garfia": 2290, "Novak Djokovic": 2410, "Daniil Medvedev": 2237,
    "Jannik Sinner": 2126, "Christopher Eubanks": 1863, "Roman Safiullin": 1817,
    "Andrey Rublev": 2130, "Holger Vitus Nodskov Rune": 2133, "Hubert Hurkacz": 2018,
    "Stefanos Tsitsipas": 2138, "Jiri Lehecka": 1927, "Matteo Berrettini": 2071,
    "Alexander Bublik": 1934, "Daniel Elahi Galan Riveros": 1701, "Denis Shapovalov": 1948,
    "Grigor Dimitrov": 2148
}
df_2023['grass_elo'] = df_2023['player'].map(grass_elo_2023)

pre_wimbledon_form_2023 = {
    "Carlos Alcaraz Garfia": 5, "Novak Djokovic": 0, "Daniil Medvedev": 2,
    "Jannik Sinner": 2, "Christopher Eubanks": 1, "Roman Safiullin": 1,
    "Andrey Rublev": 4, "Holger Vitus Nodskov Rune": 3, "Hubert Hurkacz": 1,
    "Stefanos Tsitsipas": 1, "Jiri Lehecka": 1, "Matteo Berrettini": 0,
    "Alexander Bublik": 5, "Daniel Elahi Galan Riveros": 0, "Denis Shapovalov": 1,
    "Grigor Dimitrov": 2
}
df_2023['pre_wimbledon_form'] = df_2023['player'].map(pre_wimbledon_form_2023)

print(f"✅ 2023 data loaded: {df_2023.shape}")

In [ ]:
# SECTION 6 — HISTORICAL DATA: 2024
# ===============================================================================
players_2024 = [
    "Alexander Zverev", "Roberto Bautista Agut", "Ben Shelton",
    "Giovanni Mpetshi Perricard", "Lorenzo Musetti", "Novak Djokovic",
    "Daniil Medvedev", "Taylor Harry Fritz", "Jannik Sinner",
    "Carlos Alcaraz Garfia", "Tommy Paul", "Ugo Humbert",
    "Arthur Fils", "Holger Vitus Nodskov Rune", "Grigor Dimitrov",
    "Alex De Minaur"
]
df_2024 = pd.DataFrame({'player': players_2024})

df_2024['first_serve_pct'] = df_2024['player'].map({
    "Alexander Zverev": 73.88, "Roberto Bautista Agut": 71.10, "Ben Shelton": 69.33,
    "Giovanni Mpetshi Perricard": 68.75, "Lorenzo Musetti": 68.74, "Novak Djokovic": 67.47,
    "Daniil Medvedev": 67.47, "Taylor Harry Fritz": 63.75, "Jannik Sinner": 63.68,
    "Carlos Alcaraz Garfia": 63.32, "Tommy Paul": 62.85, "Ugo Humbert": 62.71,
    "Arthur Fils": 62.48, "Holger Vitus Nodskov Rune": 61.28, "Grigor Dimitrov": 58.97,
    "Alex De Minaur": 53.00
})
df_2024['first_serve_won_pct'] = df_2024['player'].map({
    "Grigor Dimitrov": 88.59, "Alexander Zverev": 85.00, "Giovanni Mpetshi Perricard": 81.21,
    "Taylor Harry Fritz": 81.05, "Jannik Sinner": 80.44, "Holger Vitus Nodskov Rune": 79.08,
    "Novak Djokovic": 78.72, "Ben Shelton": 77.24, "Ugo Humbert": 77.03,
    "Alex De Minaur": 75.00, "Tommy Paul": 75.00, "Daniil Medvedev": 74.22,
    "Carlos Alcaraz Garfia": 73.55, "Arthur Fils": 72.56, "Lorenzo Musetti": 71.68,
    "Roberto Bautista Agut": 67.36
})
df_2024['second_serve_won_pct'] = df_2024['player'].map({
    "Taylor Harry Fritz": 65.13, "Holger Vitus Nodskov Rune": 59.60, "Lorenzo Musetti": 59.32,
    "Ugo Humbert": 59.09, "Grigor Dimitrov": 57.81, "Alexander Zverev": 57.58,
    "Tommy Paul": 56.72, "Carlos Alcaraz Garfia": 56.40, "Roberto Bautista Agut": 56.20,
    "Jannik Sinner": 56.04, "Novak Djokovic": 55.85, "Daniil Medvedev": 55.35,
    "Giovanni Mpetshi Perricard": 54.67, "Ben Shelton": 54.34, "Alex De Minaur": 51.68,
    "Arthur Fils": 48.22
})
df_2024['serve_pts_won_pct'] = df_2024['player'].map({
    "Alexander Zverev": 77.84, "Grigor Dimitrov": 75.96, "Taylor Harry Fritz": 75.28,
    "Giovanni Mpetshi Perricard": 72.92, "Jannik Sinner": 71.58, "Holger Vitus Nodskov Rune": 71.54,
    "Novak Djokovic": 71.28, "Ugo Humbert": 70.34, "Ben Shelton": 70.21,
    "Tommy Paul": 68.21, "Daniil Medvedev": 68.08, "Lorenzo Musetti": 67.81,
    "Carlos Alcaraz Garfia": 67.26, "Roberto Bautista Agut": 64.14, "Alex De Minaur": 64.04,
    "Arthur Fils": 63.43
})
df_2024['ace_pct'] = df_2024['player'].map({
    "Giovanni Mpetshi Perricard": 23.96, "Alexander Zverev": 16.89, "Taylor Harry Fritz": 14.50,
    "Grigor Dimitrov": 13.46, "Holger Vitus Nodskov Rune": 13.08, "Ugo Humbert": 11.86,
    "Novak Djokovic": 10.73, "Jannik Sinner": 10.70, "Daniil Medvedev": 10.44,
    "Ben Shelton": 9.75, "Alex De Minaur": 8.20, "Carlos Alcaraz Garfia": 7.99,
    "Tommy Paul": 7.58, "Arthur Fils": 7.05, "Lorenzo Musetti": 5.43,
    "Roberto Bautista Agut": 3.38
})
df_2024['double_fault_pct'] = df_2024['player'].map({
    "Giovanni Mpetshi Perricard": 5.63, "Arthur Fils": 4.95, "Daniil Medvedev": 4.84,
    "Grigor Dimitrov": 3.85, "Holger Vitus Nodskov Rune": 3.85, "Alex De Minaur": 3.47,
    "Carlos Alcaraz Garfia": 3.43, "Ben Shelton": 3.37, "Novak Djokovic": 3.11,
    "Alexander Zverev": 2.37, "Roberto Bautista Agut": 2.11, "Jannik Sinner": 1.93,
    "Tommy Paul": 1.66, "Ugo Humbert": 1.27, "Lorenzo Musetti": 1.19,
    "Taylor Harry Fritz": 1.12
})
df_2024['bp_saved_pct'] = df_2024['player'].map({
    "Giovanni Mpetshi Perricard": 79.31, "Alexander Zverev": 77.78, "Ben Shelton": 74.19,
    "Holger Vitus Nodskov Rune": 74.07, "Tommy Paul": 70.21, "Roberto Bautista Agut": 68.29,
    "Lorenzo Musetti": 66.00, "Arthur Fils": 64.71, "Jannik Sinner": 64.29,
    "Novak Djokovic": 64.29, "Daniil Medvedev": 62.16, "Taylor Harry Fritz": 61.90,
    "Alex De Minaur": 59.26, "Grigor Dimitrov": 58.33, "Carlos Alcaraz Garfia": 56.52,
    "Ugo Humbert": 56.52
})
df_2024['return_pts_won_pct'] = df_2024['player'].map({
    "Alex De Minaur": 47.53, "Carlos Alcaraz Garfia": 40.69, "Tommy Paul": 39.30,
    "Lorenzo Musetti": 38.85, "Novak Djokovic": 38.46, "Ugo Humbert": 38.09,
    "Grigor Dimitrov": 37.57, "Roberto Bautista Agut": 37.32, "Jannik Sinner": 36.95,
    "Daniil Medvedev": 36.21, "Taylor Harry Fritz": 34.85, "Arthur Fils": 34.77,
    "Holger Vitus Nodskov Rune": 34.04, "Alexander Zverev": 33.95, "Ben Shelton": 30.43,
    "Giovanni Mpetshi Perricard": 29.41
})
df_2024['second_serve_return_won_pct'] = df_2024['player'].map({
    "Alex De Minaur": 58.38, "Roberto Bautista Agut": 58.33, "Jannik Sinner": 56.84,
    "Taylor Harry Fritz": 53.59, "Novak Djokovic": 53.33, "Grigor Dimitrov": 52.48,
    "Daniil Medvedev": 51.98, "Lorenzo Musetti": 51.77, "Carlos Alcaraz Garfia": 50.87,
    "Ugo Humbert": 50.24, "Tommy Paul": 46.97, "Holger Vitus Nodskov Rune": 46.63,
    "Arthur Fils": 44.44, "Alexander Zverev": 44.00, "Ben Shelton": 43.14,
    "Giovanni Mpetshi Perricard": 38.89
})
df_2024['bp_won_pct'] = df_2024['player'].map({
    "Jannik Sinner": 52.94, "Grigor Dimitrov": 50.00, "Tommy Paul": 50.00,
    "Daniil Medvedev": 47.73, "Carlos Alcaraz Garfia": 44.09, "Alexander Zverev": 43.75,
    "Giovanni Mpetshi Perricard": 43.48, "Roberto Bautista Agut": 39.47, "Alex De Minaur": 38.46,
    "Taylor Harry Fritz": 37.21, "Lorenzo Musetti": 36.71, "Ugo Humbert": 36.00,
    "Novak Djokovic": 35.00, "Arthur Fils": 34.04, "Holger Vitus Nodskov Rune": 32.43,
    "Ben Shelton": 28.57
})
df_2024['tiebreak_won_pct'] = df_2024['player'].map({
    "Alex De Minaur": 100.00, "Jannik Sinner": 83.33, "Carlos Alcaraz Garfia": 80.00,
    "Alexander Zverev": 66.67, "Daniil Medvedev": 66.67, "Novak Djokovic": 66.67,
    "Roberto Bautista Agut": 66.67, "Ugo Humbert": 66.67, "Giovanni Mpetshi Perricard": 60.00,
    "Holger Vitus Nodskov Rune": 50.00, "Lorenzo Musetti": 50.00, "Taylor Harry Fritz": 50.00,
    "Ben Shelton": 40.00, "Tommy Paul": None, "Arthur Fils": None,
    "Grigor Dimitrov": None
})
df_2024['points_dominance'] = df_2024['player'].map({
    "Grigor Dimitrov": 1.563, "Alexander Zverev": 1.532, "Taylor Harry Fritz": 1.410,
    "Novak Djokovic": 1.339, "Alex De Minaur": 1.322, "Jannik Sinner": 1.300,
    "Ugo Humbert": 1.284, "Carlos Alcaraz Garfia": 1.243, "Tommy Paul": 1.236,
    "Lorenzo Musetti": 1.207, "Holger Vitus Nodskov Rune": 1.196, "Daniil Medvedev": 1.134,
    "Giovanni Mpetshi Perricard": 1.086, "Roberto Bautista Agut": 1.040, "Ben Shelton": 1.021,
    "Arthur Fils": 0.951
})
df_2024['point_time'] = df_2024['player'].map({
    "Novak Djokovic": 45.48, "Carlos Alcaraz Garfia": 44.02, "Jannik Sinner": 43.72,
    "Ugo Humbert": 42.63, "Lorenzo Musetti": 42.33, "Alexander Zverev": 42.05,
    "Arthur Fils": 42.02, "Roberto Bautista Agut": 41.89, "Alex De Minaur": 41.28,
    "Daniil Medvedev": 41.15, "Tommy Paul": 40.61, "Grigor Dimitrov": 40.21,
    "Ben Shelton": 40.07, "Taylor Harry Fritz": 39.48, "Holger Vitus Nodskov Rune": 38.82,
    "Giovanni Mpetshi Perricard": 33.67
})
df_2024['round_reached'] = df_2024['player'].map({
    "Carlos Alcaraz Garfia": "W", "Novak Djokovic": "F", "Lorenzo Musetti": "SF",
    "Daniil Medvedev": "SF", "Alexander Zverev": "R16", "Taylor Harry Fritz": "QF",
    "Tommy Paul": "QF", "Giovanni Mpetshi Perricard": "R16", "Alex De Minaur": "QF",
    "Ugo Humbert": "R16", "Ben Shelton": "R16", "Jannik Sinner": "QF",
    "Grigor Dimitrov": "R16", "Holger Vitus Nodskov Rune": "R16", "Roberto Bautista Agut": "R16",
    "Arthur Fils": "R16"
})
df_2024['year'] = 2024

grass_elo_2024 = {
    "Carlos Alcaraz Garfia": 2304, "Novak Djokovic": 2347, "Lorenzo Musetti": 1946,
    "Daniil Medvedev": 2209, "Jannik Sinner": 2350, "Alexander Zverev": 2211,
    "Taylor Harry Fritz": 2089, "Tommy Paul": 2069, "Giovanni Mpetshi Perricard": 1785,
    "Alex De Minaur": 2151, "Ugo Humbert": 1990, "Ben Shelton": 1950,
    "Holger Vitus Nodskov Rune": 2031, "Roberto Bautista Agut": 1883, "Arthur Fils": 1860,
    "Grigor Dimitrov": 2148
}
df_2024['grass_elo'] = df_2024['player'].map(grass_elo_2024)

pre_wimbledon_form_2024 = {
    "Jannik Sinner": 5, "Alexander Zverev": 3, "Daniil Medvedev": 1,
    "Taylor Harry Fritz": 2, "Tommy Paul": 5, "Ben Shelton": 1,
    "Grigor Dimitrov": 1, "Holger Vitus Nodskov Rune": 1, "Ugo Humbert": 1,
    "Arthur Fils": 2, "Giovanni Mpetshi Perricard": 2, "Carlos Alcaraz Garfia": 2,
    "Novak Djokovic": 0, "Lorenzo Musetti": 4, "Alex De Minaur": 1,
    "Roberto Bautista Agut": 1
}
df_2024['pre_wimbledon_form'] = df_2024['player'].map(pre_wimbledon_form_2024)

print(f"✅ 2024 data loaded: {df_2024.shape}")

In [ ]:
# SECTION 7 — COMBINE ALL YEARS + BUILD TARGET
# ===============================================================================
df_all = pd.concat([df_2019, df_2021, df_2022, df_2023, df_2024], ignore_index=True)

round_mapping = {'R16': 1, 'QF': 2, 'SF': 3, 'F': 4, 'W': 5}
df_all['round_score'] = df_all['round_reached'].map(round_mapping)
df_all['reached_qf'] = (df_all['round_score'] >= 2).astype(int)

print(f"✅ df_all built: {df_all.shape}")
print(f"Years: {sorted(df_all['year'].unique())}")
print(f"Round distribution:\n{df_all['round_reached'].value_counts()}")

In [ ]:
# SECTION 8 — MATCH-LEVEL DATA (FOR HEAD-TO-HEAD MODEL TRAINING)
# ===============================================================================
matches_2019 = [
    {"winner": "Roger Federer", "loser": "Matteo Berrettini", "round": "R16", "year": 2019},
    {"winner": "Kei Nishikori", "loser": "Mikhail Kukushkin", "round": "R16", "year": 2019},
    {"winner": "Guido Pella", "loser": "Milos Raonic", "round": "R16", "year": 2019},
    {"winner": "Novak Djokovic", "loser": "Ugo Humbert", "round": "R16", "year": 2019},
    {"winner": "Sam Querrey", "loser": "Tennys Sandgren", "round": "R16", "year": 2019},
    {"winner": "Roberto Bautista Agut", "loser": "Benoit Paire", "round": "R16", "year": 2019},
    {"winner": "David Goffin", "loser": "Fernando Verdasco", "round": "R16", "year": 2019},
    {"winner": "Rafael Nadal", "loser": "Joao Sousa", "round": "R16", "year": 2019},
    {"winner": "Rafael Nadal", "loser": "Sam Querrey", "round": "QF", "year": 2019},
    {"winner": "Roger Federer", "loser": "Kei Nishikori", "round": "QF", "year": 2019},
    {"winner": "Novak Djokovic", "loser": "David Goffin", "round": "QF", "year": 2019},
    {"winner": "Roberto Bautista Agut", "loser": "Guido Pella", "round": "QF", "year": 2019},
    {"winner": "Roger Federer", "loser": "Rafael Nadal", "round": "SF", "year": 2019},
    {"winner": "Novak Djokovic", "loser": "Roberto Bautista Agut", "round": "SF", "year": 2019},
    {"winner": "Novak Djokovic", "loser": "Roger Federer", "round": "F", "year": 2019},
]

matches_2021 = [
    {"winner": "Hubert Hurkacz", "loser": "Daniil Medvedev", "round": "R16", "year": 2021},
    {"winner": "Roger Federer", "loser": "Lorenzo Sonego", "round": "R16", "year": 2021},
    {"winner": "Felix Auger Aliassime", "loser": "Alexander Zverev", "round": "R16", "year": 2021},
    {"winner": "Novak Djokovic", "loser": "Christian Garin", "round": "R16", "year": 2021},
    {"winner": "Marton Fucsovics", "loser": "Andrey Rublev", "round": "R16", "year": 2021},
    {"winner": "Denis Shapovalov", "loser": "Roberto Bautista Agut", "round": "R16", "year": 2021},
    {"winner": "Karen Khachanov", "loser": "Sebastian Korda", "round": "R16", "year": 2021},
    {"winner": "Matteo Berrettini", "loser": "Ilya Ivashka", "round": "R16", "year": 2021},
    {"winner": "Matteo Berrettini", "loser": "Felix Auger Aliassime", "round": "QF", "year": 2021},
    {"winner": "Hubert Hurkacz", "loser": "Roger Federer", "round": "QF", "year": 2021},
    {"winner": "Novak Djokovic", "loser": "Marton Fucsovics", "round": "QF", "year": 2021},
    {"winner": "Denis Shapovalov", "loser": "Karen Khachanov", "round": "QF", "year": 2021},
    {"winner": "Novak Djokovic", "loser": "Denis Shapovalov", "round": "SF", "year": 2021},
    {"winner": "Matteo Berrettini", "loser": "Hubert Hurkacz", "round": "SF", "year": 2021},
    {"winner": "Novak Djokovic", "loser": "Matteo Berrettini", "round": "F", "year": 2021},
]

matches_2022 = [
    {"winner": "Rafael Nadal", "loser": "Botic Van De Zandschulp", "round": "R16", "year": 2022},
    {"winner": "Taylor Harry Fritz", "loser": "Jason Kubler", "round": "R16", "year": 2022},
    {"winner": "Nick Kyrgios", "loser": "Brandon Nakashima", "round": "R16", "year": 2022},
    {"winner": "Christian Garin", "loser": "Alex De Minaur", "round": "R16", "year": 2022},
    {"winner": "Novak Djokovic", "loser": "Tim Van Rijthoven", "round": "R16", "year": 2022},
    {"winner": "Jannik Sinner", "loser": "Carlos Alcaraz Garfia", "round": "R16", "year": 2022},
    {"winner": "Cameron Norrie", "loser": "Tommy Paul", "round": "R16", "year": 2022},
    {"winner": "David Goffin", "loser": "Frances Tiafoe", "round": "R16", "year": 2022},
    {"winner": "Nick Kyrgios", "loser": "Christian Garin", "round": "QF", "year": 2022},
    {"winner": "Rafael Nadal", "loser": "Taylor Harry Fritz", "round": "QF", "year": 2022},
    {"winner": "Cameron Norrie", "loser": "David Goffin", "round": "QF", "year": 2022},
    {"winner": "Novak Djokovic", "loser": "Jannik Sinner", "round": "QF", "year": 2022},
    {"winner": "Nick Kyrgios", "loser": "Rafael Nadal", "round": "SF", "year": 2022},
    {"winner": "Novak Djokovic", "loser": "Cameron Norrie", "round": "SF", "year": 2022},
    {"winner": "Novak Djokovic", "loser": "Nick Kyrgios", "round": "F", "year": 2022},
]

matches_2023 = [
    {"winner": "Carlos Alcaraz Garfia", "loser": "Matteo Berrettini", "round": "R16", "year": 2023},
    {"winner": "Holger Vitus Nodskov Rune", "loser": "Grigor Dimitrov", "round": "R16", "year": 2023},
    {"winner": "Novak Djokovic", "loser": "Hubert Hurkacz", "round": "R16", "year": 2023},
    {"winner": "Christopher Eubanks", "loser": "Stefanos Tsitsipas", "round": "R16", "year": 2023},
    {"winner": "Daniil Medvedev", "loser": "Jiri Lehecka", "round": "R16", "year": 2023},
    {"winner": "Jannik Sinner", "loser": "Daniel Elahi Galan Riveros", "round": "R16", "year": 2023},
    {"winner": "Roman Safiullin", "loser": "Denis Shapovalov", "round": "R16", "year": 2023},
    {"winner": "Andrey Rublev", "loser": "Alexander Bublik", "round": "R16", "year": 2023},
    {"winner": "Carlos Alcaraz Garfia", "loser": "Holger Vitus Nodskov Rune", "round": "QF", "year": 2023},
    {"winner": "Daniil Medvedev", "loser": "Christopher Eubanks", "round": "QF", "year": 2023},
    {"winner": "Novak Djokovic", "loser": "Andrey Rublev", "round": "QF", "year": 2023},
    {"winner": "Jannik Sinner", "loser": "Roman Safiullin", "round": "QF", "year": 2023},
    {"winner": "Carlos Alcaraz Garfia", "loser": "Daniil Medvedev", "round": "SF", "year": 2023},
    {"winner": "Novak Djokovic", "loser": "Jannik Sinner", "round": "SF", "year": 2023},
    {"winner": "Carlos Alcaraz Garfia", "loser": "Novak Djokovic", "round": "F", "year": 2023},
]

matches_2024 = [
    {"winner": "Novak Djokovic", "loser": "Holger Vitus Nodskov Rune", "round": "R16", "year": 2024},
    {"winner": "Taylor Harry Fritz", "loser": "Alexander Zverev", "round": "R16", "year": 2024},
    {"winner": "Alex De Minaur", "loser": "Arthur Fils", "round": "R16", "year": 2024},
    {"winner": "Lorenzo Musetti", "loser": "Giovanni Mpetshi Perricard", "round": "R16", "year": 2024},
    {"winner": "Tommy Paul", "loser": "Roberto Bautista Agut", "round": "R16", "year": 2024},
    {"winner": "Daniil Medvedev", "loser": "Grigor Dimitrov", "round": "R16", "year": 2024},
    {"winner": "Jannik Sinner", "loser": "Ben Shelton", "round": "R16", "year": 2024},
    {"winner": "Carlos Alcaraz Garfia", "loser": "Ugo Humbert", "round": "R16", "year": 2024},
    {"winner": "Novak Djokovic", "loser": "Alex De Minaur", "round": "QF", "year": 2024},
    {"winner": "Lorenzo Musetti", "loser": "Taylor Harry Fritz", "round": "QF", "year": 2024},
    {"winner": "Carlos Alcaraz Garfia", "loser": "Tommy Paul", "round": "QF", "year": 2024},
    {"winner": "Daniil Medvedev", "loser": "Jannik Sinner", "round": "QF", "year": 2024},
    {"winner": "Novak Djokovic", "loser": "Lorenzo Musetti", "round": "SF", "year": 2024},
    {"winner": "Carlos Alcaraz Garfia", "loser": "Daniil Medvedev", "round": "SF", "year": 2024},
    {"winner": "Carlos Alcaraz Garfia", "loser": "Novak Djokovic", "round": "F", "year": 2024},
]

df_matches_2019 = pd.DataFrame(matches_2019)
df_matches_2021 = pd.DataFrame(matches_2021)
df_matches_2022 = pd.DataFrame(matches_2022)
df_matches_2023 = pd.DataFrame(matches_2023)
df_matches_2024 = pd.DataFrame(matches_2024)

df_matches_all = pd.concat([
    df_matches_2019, df_matches_2021, df_matches_2022, df_matches_2023, df_matches_2024
], ignore_index=True)

print(f"✅ Total matches: {len(df_matches_all)}")

In [ ]:
# SECTION 9 — BUILD MATCHUP DATASET (STAT DIFFERENCES) & TRAIN MODELS
# ===============================================================================
STAT_COLS = [
    'first_serve_pct', 'first_serve_won_pct', 'second_serve_won_pct',
    'serve_pts_won_pct', 'ace_pct', 'double_fault_pct', 'bp_saved_pct',
    'return_pts_won_pct', 'second_serve_return_won_pct', 'bp_won_pct',
    'tiebreak_won_pct', 'points_dominance', 'point_time',
    'grass_elo', 'pre_wimbledon_form'
]

player_stats = df_all[['player', 'year'] + STAT_COLS].copy()

df_model = df_matches_all.merge(
    player_stats.add_prefix('w_').rename(columns={'w_player': 'winner', 'w_year': 'year'}),
    on=['winner', 'year'], how='left'
)
df_model = df_model.merge(
    player_stats.add_prefix('l_').rename(columns={'l_player': 'loser', 'l_year': 'year'}),
    on=['loser', 'year'], how='left'
)

diff_df = pd.DataFrame()
for stat in STAT_COLS:
    diff_df[f'diff_{stat}'] = df_model[f'w_{stat}'] - df_model[f'l_{stat}']
diff_df['round'] = df_model['round']
diff_df['year'] = df_model['year']
diff_df['winner'] = df_model['winner']
diff_df['loser'] = df_model['loser']

rows = []
for _, row in diff_df.iterrows():
    r1 = {f'diff_{stat}': row[f'diff_{stat}'] for stat in STAT_COLS}
    r1['won'] = 1
    rows.append(r1)
    r2 = {f'diff_{stat}': -row[f'diff_{stat}'] for stat in STAT_COLS}
    r2['won'] = 0
    rows.append(r2)

df_matchup = pd.DataFrame(rows)
print(f"✅ df_matchup built: {df_matchup.shape}")

DIFF_FEATURES = [f'diff_{stat}' for stat in STAT_COLS]
y = df_matchup['won'].values

imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(df_matchup[DIFF_FEATURES])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# XGBoost — primary model
xgb_model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                           random_state=42, eval_metric='logloss')
xgb_model.fit(X_scaled, y)

# Logistic Regression — benchmark
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_scaled, y)
log_cv = cross_val_score(log_model, X_scaled, y, cv=5, scoring='roc_auc')

print(f"✅ XGBoost trained")
print(f"✅ Logistic Regression — Mean AUC: {log_cv.mean():.3f}")

In [ ]:
# SECTION 10 — SPEARMAN CORRELATIONS (BONFERRONI CORRECTED)
# ===============================================================================
results_corr = []
for stat in STAT_COLS:
    valid = df_all[['round_score', stat]].dropna()
    corr, pval = scipy_stats.spearmanr(valid[stat], valid['round_score'])
    results_corr.append({'stat': stat, 'spearman_r': round(corr, 3), 'p_value': round(pval, 4)})

corr_df = pd.DataFrame(results_corr).sort_values('spearman_r', ascending=False)
n_tests = len(corr_df)
corr_df['p_bonferroni'] = (corr_df['p_value'] * n_tests).clip(upper=1.0).round(4)
corr_df['significant'] = corr_df['p_bonferroni'] < 0.05

xgb_importance = pd.DataFrame({
    'stat': DIFF_FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("✅ Correlations computed")
print(corr_df[['stat', 'spearman_r', 'p_bonferroni', 'significant']].to_string(index=False))

In [ ]:
# SECTION 11 — MODEL VALIDATION (ROC CURVE, 5-FOLD CV)
# ===============================================================================
y_match = df_matchup['won'].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tprs, mean_fpr, aucs_list = [], np.linspace(0, 1, 100), []

for train, test in cv.split(X_scaled, y_match):
    xgb_cv = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                            random_state=42, eval_metric='logloss')
    xgb_cv.fit(X_scaled[train], y_match[train])
    probs = xgb_cv.predict_proba(X_scaled[test])[:, 1]
    fpr, tpr, _ = roc_curve(y_match[test], probs)
    tprs.append(np.interp(mean_fpr, fpr, tpr))
    aucs_list.append(auc(fpr, tpr))

mean_tpr = np.mean(tprs, axis=0)
std_tpr = np.std(tprs, axis=0)
aucs = aucs_list

print(f"✅ Mean AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

In [ ]:
# SECTION 12 — SAVE HISTORICAL DATA
# ===============================================================================
df_all.to_csv('wimbledon_data.csv', index=False)
df_matchup.to_csv('wimbledon_matchup.csv', index=False)
print("✅ Saved: wimbledon_data.csv, wimbledon_matchup.csv")

In [ ]:
# SECTION 13 — 2026 CONTENDERS: STATS (20 KEY PLAYERS)
# ===============================================================================
players_2026 = [
    "Jannik Sinner", "Alexander Zverev", "Novak Djokovic",
    "Felix Auger-Aliassime", "Ben Shelton", "Daniil Medvedev",
    "Taylor Fritz", "Alex De Minaur", "Alexander Bublik",
    "Andrey Rublev", "Jiri Lehecka", "Jack Draper",
    "Hubert Hurkacz", "Matteo Berrettini", "Grigor Dimitrov",
    "Tommy Paul", "Ugo Humbert", "Giovanni Mpetshi Perricard",
    "Flavio Cobolli", "Casper Ruud"
]
df_2026 = pd.DataFrame({'player': players_2026})

df_2026['first_serve_pct'] = [62.8,72.5,69.5,64.1,66.2,61.6,67.4,56.0,65.5,61.1,62.4,63.3,66.4,63.2,60.5,57.4,61.0,68.2,60.3,64.0]
df_2026['first_serve_won_pct'] = [80.2,81.2,77.9,79.4,77.0,75.9,80.9,71.2,78.1,78.1,81.2,78.9,81.9,75.6,82.1,77.0,76.6,81.0,73.5,75.0]
df_2026['second_serve_won_pct'] = [63.1,56.9,50.3,51.6,57.0,49.6,57.6,61.4,49.3,57.3,57.8,54.5,59.3,44.0,54.9,52.6,54.4,53.5,53.9,50.0]
df_2026['bp_saved_pct'] = [72.3,59.4,66.5,65.7,71.4,62.0,68.7,66.5,66.2,66.8,63.5,65.0,86.0,66.5,66.0,62.8,65.3,64.7,63.4,62.0]
df_2026['return_pts_won_pct'] = [33.7,28.8,31.6,29.5,24.5,33.6,25.4,31.5,27.2,29.2,27.3,35.0,32.0,28.1,36.0,30.4,25.6,22.0,28.6,28.3]
df_2026['second_serve_return_won_pct'] = [56.5,51.0,52.7,47.4,46.6,53.1,49.7,54.2,47.7,49.6,46.8,49.0,45.0,44.8,50.0,51.6,48.3,36.2,48.0,48.0]
df_2026['bp_won_pct'] = [43.2,40.4,41.9,38.1,36.3,39.6,34.2,40.9,39.1,39.3,34.9,36.0,25.0,42.7,38.0,40.6,39.1,31.9,39.4,32.0]
df_2026['tiebreak_won_pct'] = [85.7,56.4,41.7,68.9,63.3,41.4,60.7,62.1,58.6,51.5,55.0,None,None,50.0,None,60.9,58.1,57.1,59.4,60.0]
df_2026['serve_pts_won_pct'] = [73.84,74.52,69.48,69.42,70.24,65.80,73.30,66.89,68.16,70.01,72.40,69.95,74.31,63.97,71.36,66.61,67.94,72.26,65.72,66.0]
df_2026['points_dominance'] = [2.19,2.59,2.20,2.35,2.87,1.96,2.89,2.12,2.51,2.40,2.65,2.00,2.32,2.28,1.98,2.19,2.65,3.28,2.30,2.33]
df_2026['pre_wimbledon_form'] = [1,3,0,2,2,2,4,2,1,1,1,0,2,0,0,4,3,1,1,1]
df_2026['grass_elo'] = [2750,2674,2880,2615,2639,2668,2701,2671,2651,2636,2621,2630,2661,2714,2643,2662,2618,2590,2619,2499]
df_2026['ace_pct'] = [8.9,16.9,13.3,14.0,9.75,10.44,14.5,8.2,13.2,9.26,13.3,12.0,20.05,15.96,12.4,7.58,11.86,23.96,8.0,5.0]
df_2026['double_fault_pct'] = [1.93,2.37,3.11,3.55,3.37,4.84,1.12,3.47,8.58,1.41,2.27,2.0,1.68,2.05,3.85,1.66,1.27,5.63,2.9,3.0]
df_2026['point_time'] = [43.72,42.05,45.48,44.32,40.07,41.15,39.48,41.28,36.95,38.42,39.15,42.0,38.40,41.74,42.23,40.61,42.63,33.67,40.0,42.0]

STAT_COLS_PRED = STAT_COLS  # same 15 features

stat_lookup = {}
for _, row in df_2026.iterrows():
    stat_lookup[row['player']] = row[STAT_COLS_PRED].values

# NOTE: Jack Draper withdrew from Wimbledon 2026 before the tournament (replaced
# by Dusan Lajovic in the draw). We keep his stats here for reference but he is
# removed from the draw and stat_lookup in Section 14 below.
print(f"✅ df_2026 built: {df_2026.shape}, {len(stat_lookup)} players with full stats")

In [ ]:
# SECTION 14 — WIMBLEDON 2026 DRAW (128 PLAYERS) + GRASS ELO
# Draper withdrew pre-tournament; replaced by Dusan Lajovic (confirmed lucky loser)
# ===============================================================================
wimbledon_draw = [
    ("Jannik Sinner",1),("Miomir Kecmanovic",50),
    ("Nuno Borges",48),("Tristan Boyer",191),
    ("Aleksandar Vukic",104),("Jenson Brooksby",81),
    ("Emilio Nava",86),("Ignacio Buse",34),
    ("Rafael Jodar",26),("Felix Gill",220),
    ("Denis Shapovalov",41),("Pablo Carreno Busta",71),
    ("Shintaro Mochizuki",151),("Max Basing",329),
    ("Ethan Quinn",47),("Luciano Darderi",16),
    ("Felix Auger-Aliassime",4),("Aleksandr Shevchenko",99),
    ("Adam Walton",85),("Dino Prizmic",89),
    ("Adolfo Daniel Vallejo",72),("Nicolas Mejia",165),
    ("Michael Zheng",144),("Cameron Norrie",29),
    ("Alejandro Davidovich Fokina",23),("Juan Manuel Cerundolo",42),
    ("Thiago Agustin Tirante",55),("Fabian Marozsan",53),
    ("Luca Van Assche",80),("Marton Fucsovics",76),
    ("Dalibor Svrcina",112),("Learner Tien",17),
    ("Alex De Minaur",6),("Roman Andres Burruchaga",65),
    ("Adrian Mannarino",40),("Titouan Droguet",116),
    ("Mattia Bellucci",67),("Zachary Svajda",66),
    ("Kamil Majchrzak",45),("Alejandro Tabilo",33),
    ("Karen Khachanov",22),("Billy Harris",155),
    ("Yannick Hanfmann",56),("Giovanni Mpetshi Perricard",83),
    ("Tallon Griekspoor",58),("James Duckworth",79),
    ("Mariano Navone",38),("Flavio Cobolli",10),
    ("Jakub Mensik",18),("Toby Samuel",123),
    ("Dane Sweeney",127),("Grigor Dimitrov",146),
    ("Stan Wawrinka",109),("Matteo Berrettini",51),
    ("Raphael Collignon",43),("Arthur Fils",24),
    ("Ugo Humbert",30),("Zizou Bergs",37),
    ("Sho Shimabukuro",90),("Jaime Faria",98),
    ("Damir Dzumhur",105),("Arthur Fery",114),
    ("Otto Virtanen",140),("Ben Shelton",5),
    ("Taylor Fritz",7),("Dusan Lajovic",153),  # Draper withdrew, replaced by Lajovic
    ("Patrick Kypson",113),("Mackenzie McDonald",145),
    ("Benjamin Bonzi",93),("Gabriel Diallo",88),
    ("Lorenzo Sonego",69),("Tomas Martin Etcheverry",32),
    ("Frances Tiafoe",19),("Terence Atmane",52),
    ("Vit Kopriva",64),("Jan Choinski",100),
    ("Kyrian Jacquet",139),("Vilius Gaubas",129),
    ("Thanasi Kokkinakis",491),("Alexander Bublik",11),
    ("Jiri Lehecka",14),("Alexei Popyrin",103),
    ("Alex Molcan",101),("Daniel Altmaier",61),
    ("Alex Michelsen",46),("Jacob Fearnley",159),
    ("Jaume Munar",44),("Francisco Cerundolo",21),
    ("Matteo Arnaldi",35),("Quentin Halys",95),
    ("Corentin Moutet",39),("Marcos Giron",92),
    ("Valentin Royer",75),("Harry Wendelken",202),
    ("Alexander Blockx",36),("Alexander Zverev",3),
    ("Casper Ruud",12),("Hubert Hurkacz",96),
    ("Hamad Medjedovic",68),("Sebastian Ofner",110),
    ("Soonwoo Kwon",200),("Martin Landaluce",60),
    ("Alexandre Muller",126),("Tommy Paul",25),
    ("Brandon Nakashima",31),("Jack Pinnington Jones",138),
    ("Jan-Lennard Struff",74),("Sebastian Baez",57),
    ("Camilo Ugo Carabelli",59),("Daniel Merida",84),
    ("Marin Cilic",62),("Daniil Medvedev",9),
    ("Andrey Rublev",13),("Roman Safiullin",132),
    ("Aleksandar Kovacevic",70),("Botic Van De Zandschulp",54),
    ("Jesper De Jong",73),("Rinky Hijikata",82),
    ("Roberto Bautista Agut",183),("Joao Fonseca",27),
    ("Arthur Rinderknech",28),("Oliver Tarvet",348),
    ("Marco Trungelliti",94),("Martin Damm",106),
    ("Hugo Gaston",118),("Stefanos Tsitsipas",87),
    ("Yibing Wu",102),("Novak Djokovic",8),
]

grass_elo_dict = {
    "Jannik Sinner":2750,"Miomir Kecmanovic":2561,"Nuno Borges":2551,
    "Luciano Darderi":2546,"Ethan Quinn":2498,"Ignacio Buse":2496,
    "Rafael Jodar":2400,"Felix Auger-Aliassime":2615,"Learner Tien":2498,
    "Alejandro Davidovich Fokina":2561,"Juan Manuel Cerundolo":2385,"Fabian Marozsan":2498,
    "Jakub Mensik":2541,"Cameron Norrie":2628,"Alex De Minaur":2671,
    "Flavio Cobolli":2619,"Karen Khachanov":2619,"Alejandro Tabilo":2619,
    "Yannick Hanfmann":2548,"Giovanni Mpetshi Perricard":2590,"Tallon Griekspoor":2626,
    "Arthur Fils":2596,"Ugo Humbert":2618,"Matteo Berrettini":2714,
    "Grigor Dimitrov":2643,"Joao Fonseca":2540,"Francisco Cerundolo":2650,
    "Ben Shelton":2639,"Jiri Lehecka":2621,"Alexander Bublik":2651,
    "Frances Tiafoe":2650,"Novak Djokovic":2880,"Hubert Hurkacz":2661,
    "Dusan Lajovic":2510,  # estimated — clay specialist, limited grass pedigree
    "Taylor Fritz":2701,"Alexander Zverev":2674,"Casper Ruud":2499,
    "Marin Cilic":2710,"Daniil Medvedev":2668,"Tommy Paul":2662,
    "Brandon Nakashima":2603,"Andrey Rublev":2636,"Roberto Bautista Agut":2643,
    "Denis Shapovalov":2602,"Stefanos Tsitsipas":2604,"Lorenzo Sonego":2604,
    "Marcos Giron":2603,"Alex Michelsen":2598,"Adrian Mannarino":2598,
    "Marton Fucsovics":2615,"Stan Wawrinka":2606,"Zizou Bergs":2568,
    "Botic Van De Zandschulp":2574,"Damir Dzumhur":2594,"Roman Safiullin":2582,
    "Mattia Bellucci":2561,"Daniel Altmaier":2595,"Alexei Popyrin":2530,
    "Quentin Halys":2539,"Arthur Rinderknech":2544,"Raphael Collignon":2547,
    "Arthur Fery":2547,"Alex Molcan":2541,"Alexander Blockx":2498,
    "Tomas Martin Etcheverry":2532,"Kamil Majchrzak":2620,"Vit Kopriva":2498,
    "Jan-Lennard Struff":2600,"Rinky Hijikata":2546,"Jaume Munar":2543,
    "Pablo Carreno Busta":2531,"Corentin Moutet":2590,"Jan Choinski":2538,
}

def estimate_grass_elo(atp_rank):
    """Fallback Grass Elo estimate for players without a known UTS rating,
    calibrated against known reference points across rank tiers."""
    if atp_rank is None: return 2480
    if atp_rank <= 10: return max(2600, int(2820 - atp_rank*15))
    if atp_rank <= 20: return max(2580, int(2750 - atp_rank*8))
    if atp_rank <= 50: return max(2540, int(2700 - atp_rank*5))
    if atp_rank <= 100: return max(2500, int(2660 - atp_rank*3))
    if atp_rank <= 200: return max(2460, int(2580 - atp_rank*1.5))
    return max(2420, int(2500 - atp_rank*0.5))

draw_with_elo = []
for player, rank in wimbledon_draw:
    elo = grass_elo_dict.get(player, estimate_grass_elo(rank))
    draw_with_elo.append((player, rank, elo))

elo_dict = {p: e for p, r, e in draw_with_elo}
draw_players = [p for p, r, e in draw_with_elo]

# Draper withdrew — remove from stat_lookup (Lajovic has no detailed 2026 stats,
# so he is handled via Grass Elo only in the hybrid model below)
if "Jack Draper" in stat_lookup:
    del stat_lookup["Jack Draper"]

print(f"✅ Draw loaded: {len(draw_players)} players")
print(f"✅ Lajovic (Fritz's new R1 opponent) Grass Elo: {elo_dict['Dusan Lajovic']}")

In [ ]:
# SECTION 15 — HYBRID MATCH PREDICTION MODEL (XGBOOST + GRASS ELO)
# ===============================================================================
def get_match_prob(player_a, player_b):
    """
    Returns probability that player_a beats player_b.
    - Both players have full 2026 stats -> 70% XGBoost + 30% Elo
    - One player has full stats         -> 40% XGBoost + 60% Elo
    - Neither has full stats            -> pure Elo
    """
    elo_a = elo_dict.get(player_a, 2500)
    elo_b = elo_dict.get(player_b, 2500)
    elo_prob = 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

    has_a = player_a in stat_lookup
    has_b = player_b in stat_lookup

    if has_a and has_b:
        stats_a, stats_b = stat_lookup[player_a], stat_lookup[player_b]
        diffs = [0 if (pd.isna(stats_a[i]) or pd.isna(stats_b[i]))
                 else float(stats_a[i]) - float(stats_b[i])
                 for i in range(len(STAT_COLS_PRED))]
        xgb_prob = xgb_model.predict_proba(scaler.transform([diffs]))[0][1]
        prob = 0.70 * xgb_prob + 0.30 * elo_prob

    elif has_a:
        hist_mean = df_all[STAT_COLS_PRED].mean().values
        stats_a = stat_lookup[player_a]
        diffs = [0 if pd.isna(stats_a[i]) else float(stats_a[i]) - float(hist_mean[i])
                 for i in range(len(STAT_COLS_PRED))]
        xgb_prob = xgb_model.predict_proba(scaler.transform([diffs]))[0][1]
        prob = 0.40 * xgb_prob + 0.60 * elo_prob

    elif has_b:
        hist_mean = df_all[STAT_COLS_PRED].mean().values
        stats_b = stat_lookup[player_b]
        diffs = [0 if pd.isna(stats_b[i]) else float(hist_mean[i]) - float(stats_b[i])
                 for i in range(len(STAT_COLS_PRED))]
        xgb_prob = xgb_model.predict_proba(scaler.transform([diffs]))[0][1]
        prob = 0.40 * (1 - xgb_prob) + 0.60 * elo_prob

    else:
        prob = elo_prob

    return np.clip(prob, 0.05, 0.95)

# Pre-compute all pairwise probabilities once (fast simulation lookup)
print("Pre-computing match probabilities...")
prob_cache = {}
for p1 in draw_players:
    for p2 in draw_players:
        if p1 != p2 and (p1, p2) not in prob_cache:
            prob = get_match_prob(p1, p2)
            prob_cache[(p1, p2)] = prob
            prob_cache[(p2, p1)] = 1 - prob
print(f"✅ Cached {len(prob_cache)} match probabilities")

print(f"\nSanity check — Fritz vs Lajovic (R1): {prob_cache[('Taylor Fritz','Dusan Lajovic')]*100:.1f}% Fritz")
print(f"Sanity check — Sinner vs Djokovic: {prob_cache[('Jannik Sinner','Novak Djokovic')]*100:.1f}% Sinner")

In [ ]:
# SECTION 16 — MONTE CARLO SIMULATION (10,000 ITERATIONS, FULL 128-PLAYER BRACKET)
# ===============================================================================
def simulate_tournament_fast(players):
    round_reached = {p: 'R128' for p in players}
    current = players.copy()
    for rname in ['R64', 'R32', 'R16', 'QF', 'SF', 'F', 'W']:
        next_round = []
        for i in range(0, len(current), 2):
            p_a, p_b = current[i], current[i+1]
            winner = p_a if random.random() < prob_cache[(p_a, p_b)] else p_b
            round_reached[winner] = rname
            next_round.append(winner)
        current = next_round
    return round_reached

N_SIMULATIONS = 10000
random.seed(42)

wins, finals, sfs, qfs, r16s = ({p: 0 for p in draw_players} for _ in range(5))

print("Running 10,000 simulations...")
for _ in range(N_SIMULATIONS):
    rd = simulate_tournament_fast(draw_players)
    for p, r in rd.items():
        if r == 'W': wins[p] += 1
        elif r == 'F': finals[p] += 1
        elif r == 'SF': sfs[p] += 1
        elif r == 'QF': qfs[p] += 1
        elif r == 'R16': r16s[p] += 1
print("✅ Simulation complete")

summary = []
for p in draw_players:
    w = wins[p] / N_SIMULATIONS * 100
    f = (wins[p] + finals[p]) / N_SIMULATIONS * 100
    sf = (wins[p] + finals[p] + sfs[p]) / N_SIMULATIONS * 100
    qf = (wins[p] + finals[p] + sfs[p] + qfs[p]) / N_SIMULATIONS * 100
    summary.append({
        'player': p, 'grass_elo': elo_dict[p],
        'win_pct': round(w, 1), 'final_pct': round(f, 1),
        'sf_pct': round(sf, 1), 'qf_pct': round(qf, 1),
    })

df_monte = pd.DataFrame(summary).sort_values('win_pct', ascending=False).reset_index(drop=True)
df_monte.to_csv('wimbledon_2026_predictions.csv', index=False)

print("\n🎾 WIMBLEDON 2026 — MONTE CARLO PREDICTIONS (TOP 20)")
print(f"{'Player':<30} | {'Win%':>5} | {'Final%':>6} | {'SF%':>5} | {'QF%':>5}")
print("-" * 60)
for _, row in df_monte.head(20).iterrows():
    print(f"{row['player']:<30} | {row['win_pct']:>5}% | {row['final_pct']:>5}% | "
          f"{row['sf_pct']:>5}% | {row['qf_pct']:>5}%")
print(f"\nWin% sum check: {df_monte['win_pct'].sum():.1f}% (should be ~100%)")
print("✅ Saved: wimbledon_2026_predictions.csv")

In [ ]:
# SECTION 17 — R1 MATCHUPS FOR KEY CONTENDERS
# ===============================================================================
r1_matchups = [
    ("Jannik Sinner", "Miomir Kecmanovic"),
    ("Felix Auger-Aliassime", "Aleksandr Shevchenko"),
    ("Alex De Minaur", "Roman Andres Burruchaga"),
    ("Ben Shelton", "Otto Virtanen"),
    ("Taylor Fritz", "Dusan Lajovic"),          # updated post-Draper-withdrawal
    ("Alexander Bublik", "Thanasi Kokkinakis"),
    ("Jiri Lehecka", "Alexei Popyrin"),
    ("Flavio Cobolli", "Mariano Navone"),
    ("Ugo Humbert", "Zizou Bergs"),
    ("Grigor Dimitrov", "Dane Sweeney"),
    ("Matteo Berrettini", "Stan Wawrinka"),
    ("Novak Djokovic", "Yibing Wu"),
    ("Daniil Medvedev", "Marin Cilic"),
    ("Andrey Rublev", "Roman Safiullin"),
    ("Casper Ruud", "Hubert Hurkacz"),
    ("Tommy Paul", "Alexandre Muller"),
    ("Alexander Zverev", "Alexander Blockx"),
    ("Frances Tiafoe", "Terence Atmane"),
    ("Giovanni Mpetshi Perricard", "Yannick Hanfmann"),
]

r1_results = []
for contender, opponent in r1_matchups:
    prob = get_match_prob(contender, opponent)
    if prob >= 0.75: rating = "✅ SAFE"
    elif prob >= 0.60: rating = "🟡 LIKELY"
    elif prob >= 0.50: rating = "🟠 CLOSE"
    else: rating = "🔴 UPSET ALERT"
    r1_results.append({'contender': contender, 'opponent': opponent,
                        'win_prob': round(prob*100, 1), 'rating': rating})

df_r1 = pd.DataFrame(r1_results).sort_values('win_prob', ascending=True)
print("🎾 R1 MATCHUP PROBABILITIES")
print(df_r1.to_string(index=False))

In [ ]:
# SECTION 18 — DRAW LUCK ANALYSIS
# ===============================================================================
luck_results = []
for contender, opponent in r1_matchups:
    actual_prob = get_match_prob(contender, opponent)
    all_probs = [get_match_prob(contender, p) for p in draw_players if p != contender]
    avg_prob = np.mean(all_probs)
    luck_score = actual_prob - avg_prob
    tourney_win = df_monte[df_monte['player'] == contender]['win_pct'].values
    luck_results.append({
        'contender': contender, 'opponent': opponent,
        'actual_prob': round(actual_prob*100, 1),
        'avg_prob': round(avg_prob*100, 1),
        'luck_score': round(luck_score*100, 1),
        'tournament_win': tourney_win[0] if len(tourney_win) else 0
    })

df_luck = pd.DataFrame(luck_results).sort_values('luck_score', ascending=False)
print("\n🎯 DRAW LUCK SCORES")
print(df_luck.to_string(index=False))
print(f"\n🍀 Luckiest: {df_luck.iloc[0]['contender']} ({df_luck.iloc[0]['luck_score']:+.1f}%)")
print(f"😤 Unluckiest: {df_luck.iloc[-1]['contender']} ({df_luck.iloc[-1]['luck_score']:+.1f}%)")

In [ ]:
# SECTION 19 — CHART 1: WHAT ACTUALLY PREDICTS WIMBLEDON SUCCESS?
# ===============================================================================
def create_correlation_chart(bg_style='dark'):
    if bg_style == 'dark':
        bg_color, paper_color, text_color = '#1a1a2e', '#1a1a2e', 'white'
        grid_color, sig_color, nosig_color, neg_color = 'rgba(255,255,255,0.1)', '#00b04f', '#444466', '#e05555'
    else:
        bg_color, paper_color, text_color = 'white', 'white', '#111111'
        grid_color, sig_color, nosig_color, neg_color = 'rgba(0,0,0,0.08)', '#006633', '#bbbbbb', '#cc3333'

    stat_labels = {
        'grass_elo': 'Grass Elo Rating', 'return_pts_won_pct': 'Return Points Won %',
        'point_time': 'Point Duration (seconds)', 'points_dominance': 'Points Dominance',
        'second_serve_return_won_pct': '2nd Serve Return Won %', 'tiebreak_won_pct': 'Tiebreak Win %',
        'second_serve_won_pct': '2nd Serve Won %', 'serve_pts_won_pct': 'Service Points Won %',
        'pre_wimbledon_form': 'Pre-Wimbledon Form', 'first_serve_pct': '1st Serve %',
        'bp_won_pct': 'Break Points Won %', 'first_serve_won_pct': '1st Serve Won %',
        'bp_saved_pct': 'Break Points Saved %', 'double_fault_pct': 'Double Fault %', 'ace_pct': 'Ace %',
    }
    corr_sorted = corr_df.sort_values('spearman_r', ascending=True)
    labels = [stat_labels[s] for s in corr_sorted['stat']]
    values = corr_sorted['spearman_r'].tolist()
    colors = [sig_color if r['significant'] else (neg_color if r['spearman_r'] < 0 else nosig_color)
              for _, r in corr_sorted.iterrows()]
    hover_texts = [f"<b>{stat_labels[r['stat']]}</b><br>Spearman r = {r['spearman_r']}<br>"
                   f"p (Bonferroni) = {r['p_bonferroni']}<br>{'✅ Significant' if r['significant'] else '❌ Not significant'}"
                   for _, r in corr_sorted.iterrows()]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=values, y=labels, orientation='h', marker_color=colors, marker_line_width=0,
                          hovertemplate='%{customdata}<extra></extra>', customdata=hover_texts))
    fig.add_annotation(x=0.52, y='Grass Elo Rating', ax=0.55, ay='Grass Elo Rating', axref='x', ayref='y',
                        text="⭐ STRONGEST PREDICTOR<br>Historical grass performance<br>dominates all other stats",
                        showarrow=True, arrowhead=2, arrowcolor='#f0a500', arrowwidth=2,
                        xanchor='left', bgcolor='#f0a500', font=dict(color='white', size=10), borderpad=5)
    fig.add_annotation(x=0.365, y='Return Points Won %', ax=0.55, ay='Points Dominance', axref='x', ayref='y',
                        text="🎾 KEY INSIGHT<br>Winning on return matters<br>more than winning on serve",
                        showarrow=True, arrowhead=2, arrowcolor='#4fc3f7', arrowwidth=2,
                        xanchor='left', bgcolor='#00639e', font=dict(color='white', size=10), borderpad=5)
    fig.add_annotation(x=0.301, y='Tiebreak Win %', ax=0.55, ay='Tiebreak Win %', axref='x', ayref='y',
                        text="💪 PRESSURE STAT<br>Mental toughness under<br>pressure is underrated",
                        showarrow=True, arrowhead=2, arrowcolor='#9b59b6', arrowwidth=2,
                        xanchor='left', bgcolor='#6c3483', font=dict(color='white', size=10), borderpad=5)
    fig.add_annotation(x=0.55, y='Ace %', text="▲ MYTH BUSTED<br>Ace rate is negatively correlated<br>Hitting aces ≠ winning matches",
                        showarrow=False, xanchor='left', bgcolor='#c0392b', font=dict(color='white', size=10), borderpad=5)
    fig.add_annotation(x=-0.26, y='Point Duration (seconds)', text="STRONG PREDICTORS", showarrow=False,
                        font=dict(color=sig_color, size=9, family='Arial Black'), xanchor='left')
    fig.add_annotation(x=-0.26, y='Pre-Wimbledon Form', text="WEAKER / NOT SIGNIFICANT", showarrow=False,
                        font=dict(color=nosig_color, size=9, family='Arial Black'), xanchor='left')
    fig.add_vline(x=0, line_width=1, line_color=text_color, opacity=0.4)
    fig.add_vline(x=0.3, line_width=1, line_dash='dash', line_color=text_color, opacity=0.25)
    fig.add_annotation(x=0.31, y=0, yref='paper', text='r = 0.30', showarrow=False,
                        font=dict(color=text_color, size=9), opacity=0.5)

    fig.update_layout(
        title=dict(text='<b>What Actually Predicts Wimbledon Success?</b><br>'
                         '<sup>Spearman correlation · Bonferroni corrected · Wimbledon 2019–2024 · '
                         'n = 80 players · hover each bar for details</sup>',
                    font=dict(size=17, color=text_color), x=0.5),
        xaxis=dict(title='Spearman Correlation with Round Reached', color=text_color, gridcolor=grid_color,
                   zerolinecolor=text_color, tickfont=dict(color=text_color), range=[-0.28, 0.85]),
        yaxis=dict(color=text_color, tickfont=dict(color=text_color, size=11), gridcolor=grid_color, automargin=True),
        plot_bgcolor=bg_color, paper_bgcolor=paper_color, height=750, width=1100,
        margin=dict(l=50, r=50, t=120, b=80), showlegend=False, font=dict(color=text_color, family='Arial')
    )
    fig.write_html(f'wimbledon_correlations_{bg_style}.html')
    fig.show()
    print(f"✅ Saved: wimbledon_correlations_{bg_style}.html")

create_correlation_chart('dark')
create_correlation_chart('light')

In [ ]:
# SECTION 20 — CHART 2: WHICH STATS PREDICTED SUCCESS IN EACH WIMBLEDON?
# ===============================================================================
def create_heatmap(bg_style='dark'):
    if bg_style == 'dark':
        bg_color, paper_color, text_color = '#1a1a2e', '#1a1a2e', 'white'
    else:
        bg_color, paper_color, text_color = 'white', 'white', '#111111'

    stat_labels_short = {
        'grass_elo': 'Grass Elo', 'return_pts_won_pct': 'Return Pts Won',
        'point_time': 'Point Time', 'points_dominance': 'Pts Dominance',
        'second_serve_return_won_pct': '2nd Srv Rtn Won', 'tiebreak_won_pct': 'Tiebreak Win',
        'second_serve_won_pct': '2nd Srv Won', 'serve_pts_won_pct': 'Srv Pts Won',
        'pre_wimbledon_form': 'Pre-Wimbledon', 'first_serve_pct': '1st Serve %',
        'bp_won_pct': 'BP Won', 'first_serve_won_pct': '1st Srv Won',
        'bp_saved_pct': 'BP Saved', 'double_fault_pct': 'Double Fault', 'ace_pct': 'Ace %',
    }
    years = [2019, 2021, 2022, 2023, 2024]
    stats = list(stat_labels_short.keys())

    heatmap_data = []
    for year in years:
        row = []
        for stat in stats:
            year_data = df_all[df_all['year'] == year][['round_score', stat]].dropna()
            if len(year_data) < 5:
                row.append(0)
            else:
                corr, _ = scipy_stats.spearmanr(year_data[stat], year_data['round_score'])
                row.append(round(corr, 3))
        heatmap_data.append(row)

    z = np.array(heatmap_data)
    avg_corr = z.mean(axis=0)
    sort_idx = np.argsort(avg_corr)[::-1]
    z = z[:, sort_idx]
    x_labels = [list(stat_labels_short.values())[i] for i in sort_idx]
    y_labels = [str(y) for y in years]

    hover_texts = []
    for i, year in enumerate(years):
        hover_texts.append([f"<b>{x_labels[j]}</b><br>Year: {year}<br>Spearman r = {z[i][j]}"
                             for j in range(len(sort_idx))])

    fig = go.Figure(data=go.Heatmap(
        z=z, x=x_labels, y=y_labels,
        colorscale=[[0.0,'#c0392b'],[0.35,'#e74c3c'],
                    [0.45,'#555577' if bg_style=='dark' else '#cccccc'],
                    [0.55,'#555577' if bg_style=='dark' else '#cccccc'],
                    [0.75,'#27ae60'],[1.0,'#00b04f']],
        zmid=0, text=z, texttemplate='%{text}', textfont=dict(size=11, color='white'),
        hovertemplate='%{customdata}<extra></extra>', customdata=hover_texts,
        showscale=True, xgap=3, ygap=3,
        colorbar=dict(title=dict(text='Spearman r', font=dict(color=text_color)),
                      tickfont=dict(color=text_color), len=0.8)
    ))
    fig.add_annotation(x='Grass Elo', y='2024', text="⭐ Most consistent<br>predictor across<br>all 5 years",
                        showarrow=True, arrowhead=2, arrowcolor='#f0a500', arrowwidth=2, ax=0, ay=-80,
                        bgcolor='#f0a500', font=dict(color='white', size=9), borderpad=4)
    fig.add_annotation(x='Ace %', y='2021', text="⚠️ Most unpredictable year<br>Stats flipped significantly",
                        showarrow=True, arrowhead=2, arrowcolor='#e05555', arrowwidth=2, ax=80, ay=0,
                        bgcolor='#c0392b', font=dict(color='white', size=9), borderpad=4)
    fig.add_annotation(x='Ace %', y='2019', text="▲ Ace % consistently<br>negative predictor",
                        showarrow=True, arrowhead=2, arrowcolor='#9b59b6', arrowwidth=2, ax=80, ay=0,
                        bgcolor='#6c3483', font=dict(color='white', size=9), borderpad=4)

    fig.update_layout(
        title=dict(text='<b>Which Stats Predicted Success in Each Wimbledon?</b><br>'
                         '<sup>Spearman correlation per year · Green = strong positive · Red = negative · '
                         'Wimbledon 2019–2024 · hover for details</sup>',
                    font=dict(size=17, color=text_color), x=0.5),
        xaxis=dict(color=text_color, tickfont=dict(color=text_color, size=11), tickangle=-35, type='category'),
        yaxis=dict(title='Wimbledon Year', color=text_color, tickfont=dict(color=text_color, size=13),
                   type='category', automargin=True),
        plot_bgcolor=bg_color, paper_bgcolor=paper_color, height=550, width=1100,
        margin=dict(l=80, r=150, t=150, b=120), font=dict(color=text_color, family='Arial')
    )
    fig.write_html(f'wimbledon_heatmap_{bg_style}.html')
    fig.show()
    print(f"✅ Saved: wimbledon_heatmap_{bg_style}.html")

create_heatmap('dark')
create_heatmap('light')

In [ ]:
# SECTION 21 — CHART 3: MODEL VALIDATION (ROC + AUC PER FOLD)
# ===============================================================================
def create_validation_chart(bg_style='dark'):
    if bg_style == 'dark':
        bg_color, paper_color, text_color = '#1a1a2e', '#1a1a2e', 'white'
        grid_color = 'rgba(255,255,255,0.1)'
    else:
        bg_color, paper_color, text_color = 'white', 'white', '#111111'
        grid_color = 'rgba(0,0,0,0.08)'

    fig = make_subplots(rows=1, cols=2,
                         subplot_titles=['ROC Curve (5-Fold Cross-Validation)', 'AUC Score per Fold'],
                         horizontal_spacing=0.12)

    fig.add_trace(go.Scatter(x=mean_fpr, y=mean_tpr + std_tpr, mode='lines', line=dict(width=0),
                              showlegend=False, hoverinfo='skip'), row=1, col=1)
    fig.add_trace(go.Scatter(x=mean_fpr, y=mean_tpr - std_tpr, mode='lines', line=dict(width=0),
                              fill='tonexty', fillcolor='rgba(0,180,80,0.15)',
                              showlegend=False, hoverinfo='skip'), row=1, col=1)
    fig.add_trace(go.Scatter(x=mean_fpr, y=mean_tpr, mode='lines', line=dict(color='#00b04f', width=3),
                              name=f'Mean ROC (AUC = {np.mean(aucs):.3f})',
                              hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>'), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', line=dict(color='grey', width=1, dash='dash'),
                              name='Random classifier', hoverinfo='skip'), row=1, col=1)

    colors_folds = ['#4fc3f7', '#f0a500', '#ef5350', '#ab47bc', '#26c6da']
    for i, (tpr_fold, auc_val) in enumerate(zip(tprs, aucs)):
        fig.add_trace(go.Scatter(x=mean_fpr, y=tpr_fold, mode='lines',
                                  line=dict(color=colors_folds[i], width=1, dash='dot'),
                                  name=f'Fold {i+1} (AUC={auc_val:.3f})', opacity=0.6,
                                  hovertemplate=f'Fold {i+1}<br>FPR: %{{x:.3f}}<br>TPR: %{{y:.3f}}<extra></extra>'),
                      row=1, col=1)

    fold_labels = [f'Fold {i+1}' for i in range(5)]
    fig.add_trace(go.Bar(x=fold_labels, y=aucs, marker_color=colors_folds, marker_line_width=0,
                          text=[f'{a:.3f}' for a in aucs], textposition='outside',
                          textfont=dict(color=text_color, size=11), name='AUC per fold',
                          hovertemplate='%{x}<br>AUC = %{y:.3f}<extra></extra>'), row=1, col=2)
    fig.add_hline(y=np.mean(aucs), line_dash='dash', line_color='#f0a500', line_width=2,
                  annotation_text=f'Mean AUC = {np.mean(aucs):.3f}', annotation_font_color='#f0a500', row=1, col=2)

    fig.update_layout(
        title=dict(text='<b>Model Validation: XGBoost Match Prediction</b><br>'
                         f'<sup>5-fold cross-validation · AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f} · '
                         'Wimbledon 2019–2024 · n = 150 match observations</sup>',
                    font=dict(size=17, color=text_color), x=0.5),
        plot_bgcolor=bg_color, paper_bgcolor=paper_color, height=550, width=1100,
        margin=dict(l=80, r=80, t=130, b=80),
        legend=dict(font=dict(color=text_color, size=10), bgcolor='rgba(0,0,0,0)',
                    bordercolor='rgba(255,255,255,0.2)', borderwidth=1),
        font=dict(color=text_color, family='Arial')
    )
    for col in [1, 2]:
        fig.update_xaxes(color=text_color, gridcolor=grid_color, tickfont=dict(color=text_color), row=1, col=col)
        fig.update_yaxes(color=text_color, gridcolor=grid_color, tickfont=dict(color=text_color), row=1, col=col)
    fig.update_xaxes(title='False Positive Rate', row=1, col=1)
    fig.update_yaxes(title='True Positive Rate', row=1, col=1)
    fig.update_xaxes(title='Cross-Validation Fold', row=1, col=2)
    fig.update_yaxes(title='AUC Score', range=[0, 1.1], row=1, col=2)
    fig.add_annotation(x=0.6, y=0.3, xref='x', yref='y',
                        text=f"AUC = {np.mean(aucs):.3f}<br>Significantly better<br>than random (0.5)",
                        showarrow=False, bgcolor='#00639e', font=dict(color='white', size=10), borderpad=5)

    fig.write_html(f'wimbledon_validation_{bg_style}.html')
    fig.show()
    print(f"✅ Saved: wimbledon_validation_{bg_style}.html")

create_validation_chart('dark')
create_validation_chart('light')

In [ ]:
# SECTION 22 — CHART 4: 2026 MONTE CARLO PREDICTIONS HEATMAP
# ===============================================================================
def create_monte_carlo_chart(bg_style='dark'):
    if bg_style == 'dark':
        bg_color, paper_color, text_color = '#1a1a2e', '#1a1a2e', 'white'
    else:
        bg_color, paper_color, text_color = 'white', 'white', '#111111'

    top20 = df_monte.head(20).copy().reset_index(drop=True)
    seed_map = {
        "Jannik Sinner": ("🇮🇹", 1), "Alex De Minaur": ("🇦🇺", 5), "Taylor Fritz": ("🇺🇸", 6),
        "Hubert Hurkacz": ("🇵🇱", None), "Felix Auger-Aliassime": ("🇨🇦", 3), "Marin Cilic": ("🇭🇷", None),
        "Novak Djokovic": ("🇷🇸", 7), "Frances Tiafoe": ("🇺🇸", 17), "Denis Shapovalov": ("🇨🇦", None),
        "Cameron Norrie": ("🇬🇧", None), "Jack Draper": ("🇬🇧", None), "Marton Fucsovics": ("🇭🇺", None),
        "Francisco Cerundolo": ("🇦🇷", None), "Tommy Paul": ("🇺🇸", None), "Tallon Griekspoor": ("🇳🇱", None),
        "Karen Khachanov": ("🇷🇺", 19), "Andrey Rublev": ("🇷🇺", 12), "Jiri Lehecka": ("🇨🇿", 13),
        "Roberto Bautista Agut": ("🇪🇸", None), "Alejandro Tabilo": ("🇨🇱", None),
        "Dusan Lajovic": ("🇷🇸", None),
    }
    player_labels = []
    for i, row in top20.iterrows():
        p = row['player']
        flag, seed = seed_map.get(p, ("🎾", None))
        seed_str = f" [{seed}]" if seed else ""
        player_labels.append(f"{i+1}. {flag} {p}{seed_str}")

    rounds = ['qf_pct', 'sf_pct', 'final_pct', 'win_pct']
    round_labels = ['Reach QF', 'Reach SF', 'Reach Final', '🏆 Win Title']
    z = [top20[r].tolist() for r in rounds]
    hover_texts = []
    for i, rnd in enumerate(rounds):
        row_hover = []
        for j, player in enumerate(top20['player']):
            flag, seed = seed_map.get(player, ("🎾", None))
            seed_str = f" (Seed {seed})" if seed else " (Unseeded)"
            row_hover.append(f"<b>{flag} {player}{seed_str}</b><br>━━━━━━━━━━━━━━━━━━<br>"
                              f"{round_labels[i]}: <b>{z[i][j]}%</b><br>"
                              f"Win title: <b>{top20.iloc[j]['win_pct']}%</b><br>"
                              f"Reach final: {top20.iloc[j]['final_pct']}%<br>"
                              f"Reach SF: {top20.iloc[j]['sf_pct']}%<br>"
                              f"Reach QF: {top20.iloc[j]['qf_pct']}%<br>━━━━━━━━━━━━━━━━━━<br>"
                              f"Grass Elo: {top20.iloc[j]['grass_elo']}")
        hover_texts.append(row_hover)

    fig = go.Figure(data=go.Heatmap(
        z=z, x=player_labels, y=round_labels,
        colorscale=[[0.0,'#0d0d1a' if bg_style=='dark' else '#f5f5f5'],[0.08,'#1a2744'],
                    [0.20,'#1a4a6e'],[0.35,'#0d6e6e'],[0.50,'#00b04f'],[0.70,'#f0a500'],[1.0,'#e63946']],
        text=[[f"{v}%" for v in row] for row in z], texttemplate='%{text}',
        textfont=dict(size=11, color='white'), hovertemplate='%{customdata}<extra></extra>',
        customdata=hover_texts, showscale=True, xgap=4, ygap=4,
        colorbar=dict(title=dict(text='Probability', font=dict(color=text_color, size=11)),
                      tickfont=dict(color=text_color), ticksuffix='%', len=0.7, thickness=15)
    ))
    fig.add_shape(type='line', xref='x', yref='paper', x0=4.5, x1=4.5, y0=0, y1=1,
                  line=dict(color='rgba(255,255,255,0.5)' if bg_style=='dark' else 'rgba(0,0,0,0.4)',
                            width=2, dash='dot'))
    fig.add_annotation(x=2, y=1.12, xref='x', yref='paper', text="⚡ TOP CONTENDERS", showarrow=False,
                        font=dict(color='#f0a500', size=10, family='Arial Black'), xanchor='center')
    fig.add_annotation(x=12, y=1.12, xref='x', yref='paper', text="DARK HORSES & OUTSIDERS →", showarrow=False,
                        font=dict(color='rgba(255,255,255,0.5)' if bg_style=='dark' else 'rgba(0,0,0,0.4)',
                                  size=10, family='Arial Black'), xanchor='center')

    fig.update_layout(
        title=dict(text='<b>Wimbledon 2026 — Tournament Win Probability by Round</b><br>'
                         '<sup>Monte Carlo simulation · 10,000 iterations · '
                         'Hybrid XGBoost (70%) + Grass Elo (30%) model · n = 128 players · '
                         'hover each cell for full breakdown</sup>',
                    font=dict(size=17, color=text_color), x=0.5),
        xaxis=dict(color=text_color, tickfont=dict(color=text_color, size=10), tickangle=-38, type='category'),
        yaxis=dict(color=text_color, tickfont=dict(color=text_color, size=12), type='category',
                   categoryorder='array', categoryarray=round_labels, automargin=True),
        plot_bgcolor=bg_color, paper_bgcolor=paper_color, height=500, width=1300,
        margin=dict(l=150, r=130, t=150, b=180), font=dict(color=text_color, family='Arial')
    )
    fig.write_html(f'wimbledon_monte_carlo_{bg_style}.html')
    fig.show()
    print(f"✅ Saved: wimbledon_monte_carlo_{bg_style}.html")

create_monte_carlo_chart('dark')
create_monte_carlo_chart('light')

In [ ]:
# SECTION 23 — CHART 5: R1 DRAW LUCK ANALYSIS
# ===============================================================================
def create_draw_luck_chart(bg_style='dark'):
    if bg_style == 'dark':
        bg_color, paper_color, text_color, grid_color = '#1a1a2e', '#1a1a2e', 'white', 'rgba(255,255,255,0.1)'
    else:
        bg_color, paper_color, text_color, grid_color = 'white', 'white', '#111111', 'rgba(0,0,0,0.08)'

    df_plot = df_luck.sort_values('luck_score', ascending=True).copy()
    df_plot['luck_score'] = df_plot['luck_score'].round(1)
    df_plot['actual_prob'] = df_plot['actual_prob'].round(1)
    df_plot['avg_prob'] = df_plot['avg_prob'].round(1)

    flag_map = {
        "Jannik Sinner":"🇮🇹","Alex De Minaur":"🇦🇺","Taylor Fritz":"🇺🇸","Hubert Hurkacz":"🇵🇱",
        "Felix Auger-Aliassime":"🇨🇦","Marin Cilic":"🇭🇷","Novak Djokovic":"🇷🇸","Frances Tiafoe":"🇺🇸",
        "Denis Shapovalov":"🇨🇦","Cameron Norrie":"🇬🇧","Jack Draper":"🇬🇧","Marton Fucsovics":"🇭🇺",
        "Francisco Cerundolo":"🇦🇷","Tommy Paul":"🇺🇸","Tallon Griekspoor":"🇳🇱","Karen Khachanov":"🇷🇺",
        "Andrey Rublev":"🇷🇺","Jiri Lehecka":"🇨🇿","Roberto Bautista Agut":"🇪🇸","Alejandro Tabilo":"🇨🇱",
        "Alexander Bublik":"🇰🇿","Grigor Dimitrov":"🇧🇬","Ben Shelton":"🇺🇸","Matteo Berrettini":"🇮🇹",
        "Ugo Humbert":"🇫🇷","Flavio Cobolli":"🇮🇹","Casper Ruud":"🇳🇴","Daniil Medvedev":"🌍",
        "Alexander Zverev":"🇩🇪","Giovanni Mpetshi Perricard":"🇫🇷",
    }
    labels = [f"{flag_map.get(row['contender'],'🎾')} {row['contender']}  vs  {row['opponent']}"
              for _, row in df_plot.iterrows()]
    values = df_plot['luck_score'].tolist()
    colors = ['#e63946' if v < -10 else '#f4a261' if v < 0 else '#2ec4b6' if v < 5 else '#00b04f' for v in values]
    hover_texts = [f"<b>{flag_map.get(row['contender'],'🎾')} {row['contender']}</b><br>━━━━━━━━━━━━━━━━━━<br>"
                   f"R1 Opponent: <b>{row['opponent']}</b><br>Win probability: <b>{row['actual_prob']}%</b><br>"
                   f"Expected vs avg: {row['avg_prob']}%<br>Luck score: <b>{'+' if row['luck_score']>0 else ''}{row['luck_score']}%</b><br>"
                   f"Tournament win%: {row['tournament_win']}%"
                   for _, row in df_plot.iterrows()]
    bar_text = [f"+{round(v,1)}%" if v > 0 else f"{round(v,1)}%" for v in values]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=values, y=labels, orientation='h', marker_color=colors, marker_line_width=0,
                          text=bar_text, textposition='outside', textfont=dict(color=text_color, size=10),
                          hovertemplate='%{customdata}<extra></extra>', customdata=hover_texts))
    fig.add_vline(x=0, line_width=2, line_color=text_color, opacity=0.5)
    fig.add_annotation(x=1.01, y=1.0, xref='paper', yref='paper',
                        text="🟢 Lucky draw<br>🔵 Slightly lucky<br>🟠 Slightly unlucky<br>🔴 Brutal draw",
                        showarrow=False, xanchor='left', yanchor='top',
                        bgcolor='rgba(0,0,0,0.4)' if bg_style=='dark' else 'rgba(240,240,240,0.9)',
                        font=dict(color=text_color, size=10), borderpad=8,
                        bordercolor='rgba(255,255,255,0.2)' if bg_style=='dark' else 'rgba(0,0,0,0.1)', borderwidth=1)

    fig.update_layout(
        title=dict(text='<b>Wimbledon 2026 — R1 Draw Luck Analysis</b><br>'
                         '<sup>Draw luck = actual R1 win prob minus expected win prob vs average opponent · '
                         'hover each bar for full details</sup>',
                    font=dict(size=16, color=text_color), x=0.5),
        xaxis=dict(title='Draw Luck Score (%)', color=text_color, gridcolor=grid_color,
                   tickfont=dict(color=text_color), ticksuffix='%', range=[-75, 28]),
        yaxis=dict(color=text_color, tickfont=dict(color=text_color, size=10), automargin=True),
        plot_bgcolor=bg_color, paper_bgcolor=paper_color, height=680, width=1150,
        margin=dict(l=50, r=180, t=140, b=80), showlegend=False, font=dict(color=text_color, family='Arial')
    )
    fig.write_html(f'wimbledon_draw_luck_{bg_style}.html')
    fig.show()
    print(f"✅ Saved: wimbledon_draw_luck_{bg_style}.html")

create_draw_luck_chart('dark')
create_draw_luck_chart('light')

In [ ]:
# SECTION 24 — CHART 6: R1 WATCHLIST
# ===============================================================================
def create_r1_watchlist(bg_style='dark'):
    if bg_style == 'dark':
        bg_color, paper_color, text_color, grid_color = '#1a1a2e', '#1a1a2e', 'white', 'rgba(255,255,255,0.1)'
    else:
        bg_color, paper_color, text_color, grid_color = 'white', 'white', '#111111', 'rgba(0,0,0,0.08)'

    flag_map = {
        "Taylor Fritz":"🇺🇸","Dusan Lajovic":"🇷🇸","Casper Ruud":"🇳🇴","Hubert Hurkacz":"🇵🇱",
        "Daniil Medvedev":"🌍","Marin Cilic":"🇭🇷","Matteo Berrettini":"🇮🇹","Stan Wawrinka":"🇨🇭",
        "Ugo Humbert":"🇫🇷","Zizou Bergs":"🇧🇪","Giovanni Mpetshi Perricard":"🇫🇷","Yannick Hanfmann":"🇩🇪",
    }
    # Headline matchup updated: Fritz now faces Lajovic after Draper's withdrawal
    watchlist = [
        ("Giovanni Mpetshi Perricard", "Yannick Hanfmann", "🚀 Ace battle — biggest servers clash"),
        ("Ugo Humbert", "Zizou Bergs", "🌿 Both excel on grass surface"),
        ("Matteo Berrettini", "Stan Wawrinka", "🎾 Grass legends — both returning from injury"),
        ("Daniil Medvedev", "Marin Cilic", "⚠️ High Grass Elo upset potential"),
        ("Casper Ruud", "Hubert Hurkacz", "💀 Seeded player as massive underdog"),
        ("Taylor Fritz", "Dusan Lajovic", "✅ Comfortable opener after Draper withdrawal"),
    ]
    matchup_labels, probs_p1, probs_p2, contexts, p1_names, p2_names = [], [], [], [], [], []
    for p1, p2, context in watchlist:
        prob_p1 = int(round(get_match_prob(p1, p2) * 100))
        prob_p2 = 100 - prob_p1
        matchup_labels.append(f"{flag_map.get(p1,'🎾')} {p1} vs {flag_map.get(p2,'🎾')} {p2}")
        probs_p1.append(prob_p1); probs_p2.append(prob_p2)
        contexts.append(context); p1_names.append(p1); p2_names.append(p2)

    fig = go.Figure()
    fig.add_trace(go.Bar(x=probs_p1, y=matchup_labels, orientation='h', marker_color='#00b04f', marker_line_width=0,
                          hovertemplate=[f"<b>{flag_map.get(p1_names[i],'🎾')} {p1_names[i]}</b><br>"
                                         f"Win probability: <b>{probs_p1[i]}%</b><br>{contexts[i]}<extra></extra>"
                                         for i in range(len(watchlist))]))
    fig.add_trace(go.Bar(x=probs_p2, y=matchup_labels, orientation='h', marker_color='#e63946', marker_line_width=0,
                          hovertemplate=[f"<b>{flag_map.get(p2_names[i],'🎾')} {p2_names[i]}</b><br>"
                                         f"Win probability: <b>{probs_p2[i]}%</b><br>{contexts[i]}<extra></extra>"
                                         for i in range(len(watchlist))]))
    for i in range(len(watchlist)):
        fig.add_annotation(x=2, y=matchup_labels[i], xref='x', yref='y', text=f"<b>{probs_p1[i]}%</b>",
                            showarrow=False, xanchor='left', font=dict(color='white', size=13, family='Arial Black'))
        fig.add_annotation(x=98, y=matchup_labels[i], xref='x', yref='y', text=f"<b>{probs_p2[i]}%</b>",
                            showarrow=False, xanchor='right', font=dict(color='white', size=13, family='Arial Black'))
        fig.add_annotation(x=101, y=matchup_labels[i], xref='x', yref='y', text=contexts[i],
                            showarrow=False, xanchor='left', font=dict(color=text_color, size=10), opacity=0.85)
    fig.add_vline(x=50, line_width=2, line_dash='dash', line_color=text_color, opacity=0.4)
    fig.add_annotation(x=50, y=1.04, xref='x', yref='paper', text="50/50", showarrow=False,
                        font=dict(color=text_color, size=10), opacity=0.5)

    fig.update_layout(
        barmode='stack',
        title=dict(text='<b>Wimbledon 2026 — R1 Watchlist</b><br>'
                         '<sup>6 most compelling first round matchups · Green = left player win probability · '
                         'Red = right player · hover for details</sup>',
                    font=dict(size=17, color=text_color), x=0.5),
        xaxis=dict(title='Win Probability (%)', color=text_color, gridcolor=grid_color,
                   tickfont=dict(color=text_color), ticksuffix='%', range=[0, 150], tickvals=[0,25,50,75,100]),
        yaxis=dict(color=text_color, tickfont=dict(color=text_color, size=11), automargin=True),
        plot_bgcolor=bg_color, paper_bgcolor=paper_color, height=560, width=1150,
        margin=dict(l=50, r=340, t=130, b=80), showlegend=False, font=dict(color=text_color, family='Arial')
    )
    fig.write_html(f'wimbledon_r1_watchlist_{bg_style}.html')
    fig.show()
    print(f"✅ Saved: wimbledon_r1_watchlist_{bg_style}.html")

create_r1_watchlist('dark')
create_r1_watchlist('light')

In [ ]:
# SECTION 25 — VERIFY ALL OUTPUT FILES SAVED
# ===============================================================================
import os
charts = [
    'wimbledon_correlations_dark.html', 'wimbledon_correlations_light.html',
    'wimbledon_heatmap_dark.html', 'wimbledon_heatmap_light.html',
    'wimbledon_validation_dark.html', 'wimbledon_validation_light.html',
    'wimbledon_monte_carlo_dark.html', 'wimbledon_monte_carlo_light.html',
    'wimbledon_draw_luck_dark.html', 'wimbledon_draw_luck_light.html',
    'wimbledon_r1_watchlist_dark.html', 'wimbledon_r1_watchlist_light.html',
    'wimbledon_data.csv', 'wimbledon_matchup.csv', 'wimbledon_2026_predictions.csv',
]
print("📁 FILE STATUS CHECK")
print("-" * 50)
all_good = True
for f in charts:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    print(f"{status} {f}")
    if not os.path.exists(f):
        all_good = False
print("-" * 50)
print("✅ All files saved!" if all_good else "⚠️ Some files missing")